# Learning what nano's rules do

Built from commit `a252150`. The one cortex (frozen, e-prop) reads each sentence; a readout learns by the delta
rule what it means - which dialogue act (Switchboard), and which fact about the speaker (Dialogue NLI) - and is
compared with nano's hand-written rules and a tuned logistic regression. See
`PREREGISTRATION_learned_language.md`.

In [ ]:
import os, sys, glob, json, subprocess, time, urllib.request
W = "/kaggle/working/brain"
for d in ("brain", "tests", "chat"):
    os.makedirs(os.path.join(W, d), exist_ok=True)
DATA = "/kaggle/working/data"
os.makedirs(DATA, exist_ok=True)
def fetch(url, dest):
    if not os.path.exists(dest):
        urllib.request.urlretrieve(url, dest)
    print(dest, os.path.getsize(dest))
fetch("https://github.com/cgpotts/swda/raw/master/swda.zip", DATA + "/swda.zip")
os.makedirs(DATA + "/dnli", exist_ok=True)
for split in ("train", "dev", "test"):
    fetch("https://huggingface.co/datasets/pietrolesci/dialogue_nli/resolve/main/data/" + split + "-00000-of-00001.parquet", DATA + "/dnli/" + split + "-00000-of-00001.parquet")
CORTEX = sorted(glob.glob("/kaggle/input/**/onecortex_*.pt", recursive=True))
print("cortex checkpoints:", CORTEX)
CORTEX = [c for c in CORTEX if "eprop" in os.path.basename(c)][0]
print("using", CORTEX, "| cpus", os.cpu_count())

In [ ]:
%%writefile /kaggle/working/brain/brain/__init__.py
"""A conversational brain: lexical cortex, prefrontal working memory, hippocampal map and episodic store, a
basal-ganglia gate that learns which of them answers, and sleep. No pretrained weights, no gradients."""

In [ ]:
%%writefile /kaggle/working/brain/brain/onecortex.py
"""One cortex: a recurrent language network that hears a conversation as a stream, predicts it, remembers it,
and speaks from the same tissue - learning by a local rule throughout.

Why this replaces the pipeline. The self-trained brain chose among candidates it was handed (a stretch of a page,
a line someone once said) using features written by hand, one sentence at a time, behind hand-written routing.
On open conversation it matched words instead of meaning ("yo" -> "yo-yo"), could not say anything new, forgot
what was said two turns ago, and stored "you're a terrible chatbot" as a fact. None of that is a training
problem. This file is the architecture that could fix it, built from pieces that have each earned a place:

  cortex        Two recurrent layers. Each layer's activity is predicted from the layer below *and from its own
                activity a moment ago* (temporal predictive coding; Millidge et al. 2023), so the state carries
                the conversation - "that", the topic, the joke setup - without any rule deciding what to keep.
  learning      Incremental predictive coding (Salvatori et al. 2024): at every word the activities settle to
                reduce prediction error, and at every settling step each synapse changes by its own layer's
                error times its own input. No backpropagation, and none through time: the previous state is
                treated as given, as it is for a neuron. In run 3 of the deep cortex this rule improved at every
                checkpoint while ordinary predictive coding drifted.
  hippocampus   A fast associative memory inside the stream: a delta-rule matrix that binds each moment's key to
                what came next, decays slowly, and is wiped between conversations. The top layer's prediction
                reads it. It learns within the conversation, one write per word - the mechanism that gave the only
                positive result in the MICrONS run (carrying a fast memory across a stream: +0.37 bits/char).
                Keys and values come through fixed random projections, as dentate gyrus inputs are fixed.
  Broca         The same cortex run forward: given the conversation so far and a speaker mark, it samples words
                until the turn ends. Several drafts are sampled; which one is said is a choice.

Tokens: lowercase words plus <unk>, <a>/<b> (who is speaking) and <p> (a paragraph boundary in read text).
"""
import math
import random
import re
import time

import torch
import torch.nn.functional as Fn

SPECIAL = ["<unk>", "<a>", "<b>", "<p>"]
UNK, SPK_A, SPK_B, PARA = 0, 1, 2, 3


# ------------------------------------------------------------------------------------------------ text
def words(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[.,!?;:'\"()-]", text.lower())


class Vocab:
    def __init__(self, counts, size):
        keep = [w for w, _ in counts.most_common(size) if w not in SPECIAL][: size - len(SPECIAL)]
        self.itos = SPECIAL + keep
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def ids(self, text):
        return [self.stoi.get(w, UNK) for w in words(text)]

    def dialogue(self, turns, first=SPK_A):
        """A conversation as one stream: speaker mark, then the turn, alternating."""
        out = []
        for i, turn in enumerate(turns):
            out.append(first if i % 2 == 0 else (SPK_B if first == SPK_A else SPK_A))
            out += self.ids(turn)
        return out

    def text(self, ids):
        out = []
        for i in ids:
            w = self.itos[i]
            if w in ("<a>", "<b>", "<p>"):
                continue
            out.append(w)
        s = " ".join(out)
        s = re.sub(r" ([.,!?;:)])", r"\1", s)
        s = re.sub(r"\( ", "(", s)
        s = re.sub(r" ' ", "'", s)
        return s


# ------------------------------------------------------------------------------------------------ the cortex
class OneCortex(torch.nn.Module):
    """Two recurrent layers of leaky neurons, a fast memory, and a readout.

    Each neuron has a membrane potential that carries a fraction `leak` of itself into the next word and takes the
    rest from its inputs. Higher areas integrate over longer timescales than lower ones (Honey et al. 2012;
    Chaudhuri et al. 2015), so layer 2 is slower than layer 1. With leak 0 this is exactly the run-1 cortex.
    """

    def __init__(self, vocab, embed=128, hidden=512, mem=64, memory=True, seed=0, leak=(0.0, 0.0)):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.V, self.d, self.H, self.m, self.memory = vocab, embed, hidden, mem, memory
        self.a1, self.a2 = float(leak[0]), float(leak[1])

        def w(o, i):
            return torch.nn.Parameter(torch.randn(o, i, generator=g) / math.sqrt(i))

        self.E = torch.nn.Parameter(torch.randn(vocab, embed, generator=g) * 0.1)
        self.W1, self.U1, self.b1 = w(hidden, embed), w(hidden, hidden), torch.nn.Parameter(torch.zeros(hidden))
        self.W2, self.U2, self.b2 = w(hidden, hidden), w(hidden, hidden), torch.nn.Parameter(torch.zeros(hidden))
        self.Wm = w(hidden, mem)                                  # what the hippocampal recall contributes
        self.Wo, self.bo = w(vocab, hidden), torch.nn.Parameter(torch.zeros(vocab))
        with torch.no_grad():                                     # recurrence starts gentle, so state neither dies
            self.U1.mul_(0.5)                                     # nor explodes before it has learned anything
            self.U2.mul_(0.5)
        # fixed random projections into and out of the fast memory: wiring, not learning
        self.register_buffer("Pk", torch.randn(mem, hidden, generator=g) / math.sqrt(hidden))
        self.register_buffer("Pv", torch.randn(mem, hidden, generator=g) / math.sqrt(hidden))
        self.decay, self.write = 0.98, 0.5

    @staticmethod
    def f(x):
        return torch.tanh(x)

    @staticmethod
    def df(x):
        return 1.0 - torch.tanh(x) ** 2

    # ---------------------------------------------------------------- state
    def fresh(self, n, device):
        """A quiet cortex: potentials at rest, an empty memory, and every synapse's eligibility trace at zero."""
        zH = torch.zeros(n, self.H, device=device)
        return {"v1": zH.clone(), "v2": zH.clone(), "M": torch.zeros(n, self.m, self.m, device=device),
                "k": torch.zeros(n, self.m, device=device),
                # eligibility traces: each synapse's fading memory of its own presynaptic input. In a layer every
                # neuron leaks at the same rate, so a trace depends only on the presynaptic side and is shared by
                # all of that input's synapses - one vector per input, not one number per synapse.
                "zW1": torch.zeros(n, self.d, device=device), "zU1": zH.clone(), "zb1": torch.zeros(n, 1, device=device),
                "zW2": zH.clone(), "zU2": zH.clone(), "zm": torch.zeros(n, self.m, device=device),
                "zb2": torch.zeros(n, 1, device=device)}

    def reset(self, state, rows):
        """Forget the given streams: a new conversation starts at rest, with an empty memory and no traces."""
        if rows is None or not bool(rows.any()):
            return state
        keep = (~rows).float()
        out = {}
        for k, v in state.items():
            out[k] = v * keep.view(-1, *([1] * (v.dim() - 1)))
        return out

    def recall(self, state):
        if not self.memory:
            return torch.zeros_like(state["k"])
        return torch.bmm(state["M"], state["k"][:, :, None])[:, :, 0]

    def drive1(self, e, state):
        return Fn.linear(e, self.W1, self.b1) + Fn.linear(self.f(state["v1"]), self.U1)

    def drive2(self, h1, state, r):
        return (Fn.linear(self.f(h1), self.W2, self.b2) + Fn.linear(self.f(state["v2"]), self.U2)
                + Fn.linear(r, self.Wm))

    def predict(self, e, state, r, h1=None):
        """Each layer's predicted potential: what it carries over by leak, plus what its inputs now drive."""
        mu1 = self.a1 * state["v1"] + (1 - self.a1) * self.drive1(e, state)
        h1 = mu1 if h1 is None else h1
        mu2 = self.a2 * state["v2"] + (1 - self.a2) * self.drive2(h1, state, r)
        return mu1, mu2

    @torch.no_grad()
    def traces_now(self, e, state, h1, r):
        """The traces as they stand this moment: last moment's, decayed by the neuron's leak, plus this input."""
        a1, a2 = self.a1, self.a2
        return {"zW1": a1 * state["zW1"] + (1 - a1) * e,
                "zU1": a1 * state["zU1"] + (1 - a1) * self.f(state["v1"]),
                "zb1": a1 * state["zb1"] + (1 - a1),
                "zW2": a2 * state["zW2"] + (1 - a2) * self.f(h1),
                "zU2": a2 * state["zU2"] + (1 - a2) * self.f(state["v2"]),
                "zm": a2 * state["zm"] + (1 - a2) * r,
                "zb2": a2 * state["zb2"] + (1 - a2)}

    @torch.no_grad()
    def advance(self, state, h1, h2, e=None, r=None):
        """After a word: the settled potentials become the state, the traces move on, and the hippocampus writes."""
        s2 = self.f(h2)
        new = {"v1": h1, "v2": h2}
        if e is not None:
            new.update(self.traces_now(e, state, h1, r if r is not None else self.recall(state)))
        else:
            new.update({k: state[k] for k in ("zW1", "zU1", "zb1", "zW2", "zU2", "zm", "zb2")})
        k = Fn.normalize(Fn.linear(s2, self.Pk), dim=1)
        if self.memory:
            v = Fn.linear(s2, self.Pv)
            prev = state["k"]
            got = torch.bmm(state["M"], prev[:, :, None])[:, :, 0]
            # delta rule: bind the previous moment's key to what this moment turned out to be
            M = self.decay * state["M"] + self.write * torch.bmm((v - got)[:, :, None], prev[:, None, :])
        else:
            M = state["M"]
        new.update(M=M, k=k)
        return new

    def forward_step(self, x, state):
        """Feedforward: predict the next word from this one and the state, without settling. Used to evaluate
        and to speak - with no target there is nothing to settle toward, so the prediction is the equilibrium."""
        e = self.E[x]
        r = self.recall(state)
        mu1, mu2 = self.predict(e, state, r)
        logits = Fn.linear(self.f(mu2), self.Wo, self.bo)
        return logits, mu1, mu2

    def params(self):
        return sum(p.numel() for p in self.parameters())


# ------------------------------------------------------------------------------------------------ learning
@torch.no_grad()
def local_changes(net, x, y, state, h1, h2, beta):
    """Every layer's prediction error at these potentials, the output error, and the energy per stream."""
    e = net.E[x]
    r = net.recall(state)
    mu1, mu2 = net.predict(e, state, r, h1)
    eps1, eps2 = h1 - mu1, h2 - mu2
    logits = Fn.linear(net.f(h2), net.Wo, net.bo)
    p = torch.softmax(logits, 1)
    rows = torch.arange(len(y), device=y.device)
    ce = torch.logsumexp(logits, 1) - logits[rows, y]
    p[rows, y] -= 1.0
    energy = 0.5 * ((eps1 * eps1).sum(1) + (eps2 * eps2).sum(1)) + beta * ce
    return {"eps1": eps1, "eps2": eps2, "p": p, "e": e, "r": r, "energy": energy, "ce": ce}


@torch.no_grad()
def settle(net, x, y, state, steps, rate, beta, opt=None):
    """Settle the potentials for one word; with `opt`, synapses change at every settling step (incremental PC).
    Returns the settled h1, h2 and the loss before settling."""
    e = net.E[x]
    r = net.recall(state)
    mu1, mu2 = net.predict(e, state, r)
    h1, h2 = mu1.clone(), mu2.clone()
    n = len(y)
    step = torch.full((n, 1), float(rate), device=x.device)
    cur = local_changes(net, x, y, state, h1, h2, beta)
    loss0 = float(cur["ce"].mean())
    for _ in range(steps):
        # layer 1 hears layer 2's error through W2, scaled by how much of layer 2 is driven rather than carried
        d1 = -cur["eps1"] + net.df(h1) * ((1 - net.a2) * (cur["eps2"] @ net.W2))
        d2 = -cur["eps2"] - net.df(h2) * ((beta * cur["p"]) @ net.Wo)
        t1, t2 = h1 + step * d1, h2 + step * d2
        nxt = local_changes(net, x, y, state, t1, t2, beta)
        ok = ((nxt["energy"] <= cur["energy"]) & torch.isfinite(nxt["energy"]))[:, None]
        h1, h2 = torch.where(ok, t1, h1), torch.where(ok, t2, h2)
        for k in ("eps1", "eps2", "p"):
            cur[k] = torch.where(ok, nxt[k], cur[k])
        cur["energy"] = torch.where(ok[:, 0], nxt["energy"], cur["energy"])
        step = torch.where(ok, step, step * 0.5)
        if opt is not None:
            apply(net, opt, x, state, h1, h2, cur, beta)
    return h1, h2, loss0


@torch.no_grad()
def local_grads(net, x, state, h1, h2, cur, beta):
    """Each synapse's change: its layer's error (rescaled by 1/beta) times its eligibility trace - the fading
    memory of its own input (e-prop; Bellec et al. 2020). With no leak the trace is just the current input, and
    this is exactly run 1's rule. Returned gradient-shaped (the negative of the change) for the optimiser."""
    n = len(x)
    e1, e2 = cur["eps1"] / beta, cur["eps2"] / beta
    p = cur["p"]
    z = net.traces_now(cur["e"], state, h1, cur["r"])
    s2 = net.f(h2)
    gE = torch.zeros_like(net.E)
    # the embedding is a lookup, so its synapses see only the word now; no trace is kept for them
    gE.index_add_(0, x, -((1 - net.a1) * (e1 @ net.W1)) / n)
    return {"Wo": p.T @ s2 / n, "bo": p.mean(0),
            "W2": -(e2.T @ z["zW2"]) / n, "U2": -(e2.T @ z["zU2"]) / n, "b2": -(e2 * z["zb2"]).mean(0),
            "Wm": -(e2.T @ z["zm"]) / n,
            "W1": -(e1.T @ z["zW1"]) / n, "U1": -(e1.T @ z["zU1"]) / n, "b1": -(e1 * z["zb1"]).mean(0), "E": gE}


def apply(net, opt, x, state, h1, h2, cur, beta):
    """Hand each synapse its local change and let the optimiser (per-synapse, so local) take the step."""
    g = local_grads(net, x, state, h1, h2, cur, beta)
    opt.zero_grad(set_to_none=True)
    for name, grad in g.items():
        getattr(net, name).grad = grad
    total = torch.sqrt(sum((v * v).sum() for v in g.values()))
    if not torch.isfinite(total):
        return
    if total > 1.0:
        for v in g.values():
            v.mul_(1.0 / total)
    opt.step()


def bp_chunk(net, opt, xs, ys, starts, state):
    """Backprop through the chunk (truncated BPTT): the yardstick, never part of the brain."""
    opt.zero_grad(set_to_none=True)
    loss = 0.0
    s = {k: v.detach() for k, v in state.items()}
    for t in range(xs.shape[0]):
        s = net.reset(s, starts[t])
        logits, mu1, mu2 = net.forward_step(xs[t], s)
        loss = loss + Fn.cross_entropy(logits, ys[t])
        s2 = net.f(mu2)
        k = Fn.normalize(Fn.linear(s2.detach(), net.Pk), dim=1)
        if net.memory:
            v = Fn.linear(s2.detach(), net.Pv)
            got = torch.bmm(s["M"], s["k"][:, :, None])[:, :, 0]
            M = net.decay * s["M"] + net.write * torch.bmm((v - got)[:, :, None], s["k"][:, None, :])
        else:
            M = s["M"]
        s = dict(s, v1=mu1, v2=mu2, M=M.detach(), k=k)
    loss = loss / xs.shape[0]
    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    opt.step()
    return float(loss.detach()), {k: v.detach() for k, v in s.items()}


def ipc_chunk(net, opt, xs, ys, starts, state, steps, rate, beta):
    losses = []
    for t in range(xs.shape[0]):
        state = net.reset(state, starts[t])
        h1, h2, loss0 = settle(net, xs[t], ys[t], state, steps, rate, beta, opt)
        losses.append(loss0)
        with torch.no_grad():
            state = net.advance(state, h1, h2, e=net.E[xs[t]], r=net.recall(state))
    return sum(losses) / len(losses), state


# ------------------------------------------------------------------------------------------------ streams
class Streams:
    """B parallel streams over a list of documents; each document is a token list. A stream that finishes a
    document starts the next one, and marks that step so the cortex forgets the old one."""

    def __init__(self, docs, n, seed=0):
        self.docs, self.n = docs, n
        self.rng = random.Random(seed)
        self.cur = [self._pick() for _ in range(n)]
        self.pos = [0] * n

    def _pick(self):
        return self.docs[self.rng.randrange(len(self.docs))]

    def chunk(self, T):
        xs = torch.zeros(T, self.n, dtype=torch.long)
        ys = torch.zeros(T, self.n, dtype=torch.long)
        st = torch.zeros(T, self.n, dtype=torch.bool)
        for b in range(self.n):
            for t in range(T):
                if self.pos[b] + 1 >= len(self.cur[b]):
                    self.cur[b], self.pos[b] = self._pick(), 0
                    st[t, b] = True
                if self.pos[b] == 0:
                    st[t, b] = True
                xs[t, b] = self.cur[b][self.pos[b]]
                ys[t, b] = self.cur[b][self.pos[b] + 1]
                self.pos[b] += 1
        return xs, ys, st


# ------------------------------------------------------------------------------------------------ measuring
@torch.no_grad()
def perplexity(net, docs, device, batch=256, limit=None):
    """Every next word of every document, each document read from a fresh state."""
    docs = [d for d in docs if len(d) > 1]
    total, count = 0.0, 0
    for s in range(0, len(docs), batch):
        group = docs[s:s + batch]
        L = max(len(d) for d in group)
        x = torch.full((len(group), L), -1, dtype=torch.long)
        for i, d in enumerate(group):
            x[i, :len(d)] = torch.tensor(d)
        x = x.to(device)
        state = net.fresh(len(group), device)
        for t in range(L - 1):
            live = x[:, t + 1] >= 0
            if not bool(live.any()):
                break
            logits, mu1, mu2 = net.forward_step(x[:, t].clamp_min(0), state)
            ce = Fn.cross_entropy(logits, x[:, t + 1].clamp_min(0), reduction="none")
            total += float(ce[live].sum())
            count += int(live.sum())
            state = net.advance(state, mu1, mu2)
        if limit and count >= limit:
            break
    return math.exp(total / max(1, count))


@torch.no_grad()
def score(net, context, candidates, device):
    """Mean log-probability of each candidate reply after the context - how the cortex judges a reply."""
    n = len(candidates)
    state = net.fresh(n, device)
    ctx = torch.tensor(context, device=device)
    for t in range(len(context) - 1):
        _, mu1, mu2 = net.forward_step(ctx[t].expand(n), state)
        state = net.advance(state, mu1, mu2)
    L = max(len(c) for c in candidates)
    seq = torch.full((n, L + 1), -1, dtype=torch.long, device=device)
    seq[:, 0] = ctx[-1]
    for i, c in enumerate(candidates):
        seq[i, 1:len(c) + 1] = torch.tensor(c, device=device)
    tot = torch.zeros(n, device=device)
    cnt = torch.zeros(n, device=device)
    for t in range(L):
        live = seq[:, t + 1] >= 0
        logits, mu1, mu2 = net.forward_step(seq[:, t].clamp_min(0), state)
        lp = -Fn.cross_entropy(logits, seq[:, t + 1].clamp_min(0), reduction="none")
        tot += lp * live
        cnt += live
        state = net.advance(state, mu1, mu2)
    return (tot / cnt.clamp_min(1)).tolist()


def score_lift(net, context, candidates, device, neutral=(SPK_A, SPK_B)):
    """How much the context raises each candidate's probability: log p(reply | context) - log p(reply | nothing).

    Plain mean log-probability rewards whatever is generic ("i'm fine.") after any context at all - run 1 scored
    every model near chance that way, counting included. What makes a reply *the* reply is that this context made
    it more likely than it would otherwise be."""
    with_ctx = score(net, context, candidates, device)
    without = score(net, list(neutral), candidates, device)
    return [a - b for a, b in zip(with_ctx, without)]


@torch.no_grad()
def speak(net, vocab, history_ids, device, drafts=6, max_words=30, temperature=0.8, top_p=0.9, seed=0):
    """Broca: continue the conversation as the next speaker, several drafts, the most fluent one said."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    # the person speaks as <a> and the brain as <b>; a turn ends when the cortex expects the other speaker
    me, other = SPK_B, SPK_A
    ctx = list(history_ids) + [me]
    state = net.fresh(drafts, device)
    for t in range(len(ctx) - 1):
        _, mu1, mu2 = net.forward_step(torch.tensor(ctx[t], device=device).expand(drafts), state)
        state = net.advance(state, mu1, mu2)
    cur = torch.tensor(ctx[-1], device=device).expand(drafts).clone()
    out = [[] for _ in range(drafts)]
    done = torch.zeros(drafts, dtype=torch.bool)
    logp = torch.zeros(drafts)
    for _ in range(max_words):
        logits, mu1, mu2 = net.forward_step(cur, state)
        state = net.advance(state, mu1, mu2)
        pr = torch.softmax(logits.cpu() / temperature, 1)
        pr[:, UNK] = 0
        pr[:, PARA] = 0
        srt, idx = pr.sort(1, descending=True)
        cut = srt.cumsum(1) > top_p
        cut[:, 0] = False
        srt[cut] = 0
        pick = idx.gather(1, torch.multinomial(srt / srt.sum(1, keepdim=True), 1, generator=g)).squeeze(1)
        for i in range(drafts):
            if done[i]:
                continue
            w = int(pick[i])
            if w in (other, me):
                done[i] = True
                continue
            out[i].append(w)
            logp[i] += float(torch.log(pr[i, w] + 1e-12))
        if bool(done.all()):
            break
        cur = pick.to(device)
    # an empty draft said nothing, which is the least fluent thing it could have done
    fluency = [float(logp[i]) / len(out[i]) if out[i] else -float("inf") for i in range(drafts)]
    ranked = sorted(range(drafts), key=lambda i: -fluency[i])
    return [vocab.text(out[i]) for i in ranked], [fluency[i] for i in ranked]


# ------------------------------------------------------------------------------------------------ training
def train(rule, streams, vocab_n, dev_docs, device, steps, T=48, lr=3e-5, seed=0, embed=128, hidden=512,
          memory=True, settle_steps=6, rate=0.2, beta=0.01, eval_every=500, log=print, deadline=None,
          leak=(0.0, 0.0), init_state=None, on_eval=None):
    torch.manual_seed(seed)
    net = OneCortex(vocab_n, embed, hidden, memory=memory, seed=seed, leak=leak).to(device)
    if init_state is not None:
        # a brain keeps learning from where it is; it does not start over each session
        net.load_state_dict(init_state)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    state = net.fresh(streams.n, device)
    curve, run, t0 = [], None, time.time()
    for step in range(1, steps + 1):
        xs, ys, st = streams.chunk(T)
        xs, ys, st = xs.to(device), ys.to(device), st.to(device)
        if rule == "bp":
            loss, state = bp_chunk(net, opt, xs, ys, st, state)
        else:
            loss, state = ipc_chunk(net, opt, xs, ys, st, state, settle_steps, rate, beta)
        run = loss if run is None else 0.98 * run + 0.02 * loss
        if step % eval_every == 0 or step == steps:
            ppl = perplexity(net, dev_docs, device, limit=40000)
            curve.append({"step": step, "words": step * T * streams.n, "train_loss": round(run, 4),
                          "dev_ppl": round(ppl, 2), "seconds": round(time.time() - t0)})
            log(f"  [{rule}{'' if memory else '-nomem'} leak {leak} seed {seed} lr {lr}] {curve[-1]}")
            if on_eval is not None:
                on_eval(net, curve)
            if not math.isfinite(ppl) or ppl > 1e5:
                log("  diverged - stopping")
                break
        if deadline and time.time() > deadline:
            log("  out of time - stopping here")
            break
    return net, curve

In [ ]:
%%writefile /kaggle/working/brain/brain/learn_language.py
"""Learning what nano's hand-written rules do: dialogue acts, and facts about the speaker.

    python brain/learn_language.py --task acts  --swda swda.zip --cortex onecortex_eprop_seed0.pt --out OUT
    python brain/learn_language.py --task facts --dnli DIR      --cortex onecortex_eprop_seed0.pt --out OUT

See PREREGISTRATION_learned_language.md. The one cortex reads each sentence (frozen); a sparse word code is a
fixed projection of its words; a readout layer learns by the delta rule - each synapse changes by (target -
output) x presynaptic activity - which is the striatum learning what a cortical pattern means. It is compared
with nano's hand-written rules, which it is meant to replace, and with a tuned logistic regression.
"""
import argparse
import collections
import csv
import glob
import io
import json
import math
import os
import random
import re
import sys
import time
import zipfile
import zlib

import numpy as np
import torch
import torch.nn.functional as Fn

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
for p in (ROOT, os.path.join(ROOT, "chat")):
    if p not in sys.path:
        sys.path.insert(0, p)

from brain import onecortex as O  # noqa: E402

ACTS = ["statement", "opinion", "question", "backchannel", "agree", "disagree", "thanks", "sorry", "greet", "bye",
        "meaning", "directive"]
# DAMSL base tags -> the acts nano routes on. Everything else (hedges, quotations, uninterpretable, continuations)
# is dropped rather than forced into a class it does not belong to.
SWDA = {"sd": "statement", "sv": "opinion", "qy": "question", "qw": "question", "qo": "question", "qh": "question",
        "qrr": "question", "b": "backchannel", "bk": "backchannel", "ba": "backchannel", "bh": "backchannel",
        "aa": "agree", "ny": "agree", "na": "agree", "aap": "agree", "ar": "disagree", "nn": "disagree",
        "ng": "disagree", "arp": "disagree", "ft": "thanks", "fa": "sorry", "fp": "greet", "fc": "bye",
        "br": "meaning", "ad": "directive"}
# run 1 used (0.01, 0.03, 0.1) and 4 passes; the word readouts and every facts arm peaked at 0.1, the edge -
# undertrained. Run 2 (amendment 1, post hoc) widens the grid and doubles the passes.
GRID_LR, GRID_DECAY = (0.03, 0.1, 0.3, 1.0, 3.0), (0.0, 1e-5)
EPOCHS = 8
GRID_C = (0.1, 0.3, 1.0, 3.0, 10.0)
SEEDS = 5
DIM = 1 << 18                       # sparse word code: hashed features, a fixed projection


# ------------------------------------------------------------------------------------------------ reading
def load_cortex(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    net = O.OneCortex(len(ck["itos"]), embed=cfg["embed"], hidden=cfg["hidden"], mem=cfg["mem"],
                      memory=cfg["memory"], leak=tuple(cfg.get("leak", (0.0, 0.0))))
    net.load_state_dict(ck["state"])
    net.eval().to(device)
    voc = O.Vocab.__new__(O.Vocab)
    voc.itos = list(ck["itos"])
    voc.stoi = {w: i for i, w in enumerate(voc.itos)}
    return net, voc, {"arm": ck.get("arm"), "talk_ppl": ck.get("talk_ppl"), "path": os.path.basename(path)}


@torch.no_grad()
def cortex_codes(net, voc, texts, device="cpu", batch=512, maxlen=40, tokens=False):
    """What the cortex makes of each sentence: mean and final top-layer state, and mean first-layer state.
    With tokens=True, also the top-layer state after each word (for finding which words are the object)."""
    H = net.H
    out = torch.zeros(len(texts), 3 * H)
    per_token = [None] * len(texts) if tokens else None
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for s in range(0, len(order), batch):
        chunk = order[s:s + batch]
        seqs = [[O.SPK_A] + voc.ids(texts[i])[:maxlen] for i in chunk]
        L = max(len(q) for q in seqs)
        x = torch.zeros(len(chunk), L, dtype=torch.long)
        mask = torch.zeros(len(chunk), L)
        for r, q in enumerate(seqs):
            x[r, :len(q)] = torch.tensor(q)
            mask[r, :len(q)] = 1
        x, mask = x.to(device), mask.to(device)
        state = net.fresh(len(chunk), device)
        s1 = torch.zeros(len(chunk), H, device=device)
        s2 = torch.zeros_like(s1)
        last = torch.zeros_like(s1)
        steps = []
        for t in range(L):
            _, mu1, mu2 = net.forward_step(x[:, t], state)
            state = net.advance(state, mu1, mu2)
            live = mask[:, t:t + 1]
            f1, f2 = net.f(mu1), net.f(mu2)
            s1 += f1 * live
            s2 += f2 * live
            last = torch.where(live.bool(), f2, last)
            if tokens and t > 0:
                steps.append(f2.half().cpu())
        n = mask.sum(1, keepdim=True).clamp_min(1)
        out[chunk] = torch.cat([s2 / n, last, s1 / n], 1).cpu()
        if tokens:
            st = torch.stack(steps, 1) if steps else torch.zeros(len(chunk), 0, H).half()
            for r, i in enumerate(chunk):
                per_token[i] = st[r, :len(seqs[r]) - 1].clone()
    return (out, per_token) if tokens else out


def _h(feature):
    return zlib.crc32(feature.encode("utf-8")) % DIM


def sentence_features(text):
    """The sparse word code of a sentence: its words, word pairs, how it starts, and how it ends."""
    w = O.words(text)
    f = ["w:" + x for x in w] + ["b:" + a + "_" + b for a, b in zip(w, w[1:])]
    f += ["first:" + (w[0] if w else ""), "first2:" + "_".join(w[:2]), "first3:" + "_".join(w[:3]),
          "end:" + (text.strip()[-1:] if text.strip() else ""), "len:" + str(min(len(w), 12) // 3)]
    return sorted({_h(x) for x in f})


def token_features(words, i):
    """The sparse code of one word in place: itself, its neighbours, its shape, and how the sentence starts."""
    g = lambda k: words[k] if 0 <= k < len(words) else ("<s>" if k < 0 else "</s>")
    w = words[i]
    f = ["w0:" + w, "w-1:" + g(i - 1), "w+1:" + g(i + 1), "w-2:" + g(i - 2), "w+2:" + g(i + 2),
         "p:" + g(i - 1) + "_" + w, "n:" + w + "_" + g(i + 1), "suf:" + w[-3:], "digit:" + str(w.isdigit()),
         "s0:" + g(0), "s01:" + g(0) + "_" + g(1), "pos:" + str(min(i, 6))]
    return sorted({_h(x) for x in f})


# ------------------------------------------------------------------------------------------------ the readout
class Delta:
    """One layer of synapses onto k output units, learning by the delta rule.

    Output p = softmax(W x + V s + b) over dense cortical input x and a sparse binary code s. After each
    minibatch, every synapse changes by lr x gain(class) x (target - p) x presynaptic activity, and all of
    them decay a little. No error is sent anywhere else: the cortex below is not touched.
    """

    def __init__(self, n_dense, k, lr, decay, gain=None, sparse=True, seed=0):
        self.k, self.lr, self.decay = k, lr, decay
        self.W = torch.zeros(k, n_dense) if n_dense else None
        self.V = torch.zeros(DIM, k) if sparse else None
        self.b = torch.zeros(k)
        self.gain = gain if gain is not None else torch.ones(k)
        self.g = torch.Generator().manual_seed(seed)

    def logits(self, X, S):
        z = self.b.expand(len(X) if X is not None else len(S), -1).clone()
        if self.W is not None and X is not None:
            z += X @ self.W.T
        if self.V is not None and S is not None:
            idx, off, wts = _bag(S)
            z += Fn.embedding_bag(idx, self.V, off, mode="sum", per_sample_weights=wts)
        return z

    def learn(self, X, S, y, lr):
        p = torch.softmax(self.logits(X, S), 1)
        err = -p
        err[torch.arange(len(y)), y] += 1.0
        err *= self.gain[y][:, None]
        n = len(y)
        if self.W is not None and X is not None:
            self.W.mul_(1 - self.decay).add_(err.T @ X, alpha=lr / n)
        if self.V is not None and S is not None:
            idx, off, wts = _bag(S)
            rows = torch.repeat_interleave(torch.arange(n), torch.tensor([len(s) for s in S]))
            self._since = getattr(self, "_since", 0) + 1
            if self.decay and self._since >= 50:     # the same decay, applied in 50-step installments
                self.V.mul_((1 - self.decay) ** self._since)
                self._since = 0
            self.V.index_add_(0, idx, err[rows] * wts[:, None] * (lr / n))
        self.b.add_(err.mean(0), alpha=lr)

    def fit(self, X, S, y, epochs=None, batch=256):
        epochs = epochs or EPOCHS
        n = len(y)
        steps = epochs * math.ceil(n / batch)
        t = 0
        for _ in range(epochs):
            order = torch.randperm(n, generator=self.g)
            for s in range(0, n, batch):
                ix = order[s:s + batch]
                lr = self.lr * (1 - t / steps)
                self.learn(X[ix] if X is not None else None, [S[i] for i in ix] if S is not None else None,
                           y[ix], lr)
                t += 1
        return self

    def proba(self, X, S, batch=4096):
        n = len(X) if X is not None else len(S)
        out = []
        for s in range(0, n, batch):
            out.append(torch.softmax(self.logits(X[s:s + batch] if X is not None else None,
                                                 S[s:s + batch] if S is not None else None), 1))
        return torch.cat(out)

    def state(self):
        return {"W": self.W, "V": self.V.to_sparse() if self.V is not None else None, "b": self.b,
                "k": self.k, "lr": self.lr, "decay": self.decay}


def _bag(S):
    lens = [len(s) for s in S]
    idx = torch.tensor([i for s in S for i in s], dtype=torch.long)
    off = torch.tensor([0] + list(np.cumsum(lens)[:-1]), dtype=torch.long)
    wts = torch.cat([torch.full((l,), 1.0 / math.sqrt(max(l, 1))) for l in lens]) if lens else torch.zeros(0)
    return idx, off, wts


def balanced_gain(y, k):
    counts = torch.bincount(y, minlength=k).float().clamp_min(1)
    g = counts.sum() / (k * counts)
    return g / g[y].mean()          # mean gain over the data is 1, so the grid's learning rates keep their meaning


def standardise(X, stats=None):
    if stats is None:
        stats = (X.mean(0), X.std(0).clamp_min(1e-4))
    return (X - stats[0]) / stats[1], stats


# ------------------------------------------------------------------------------------------------ scoring
def macro_f1(gold, pred, labels=None):
    from sklearn.metrics import f1_score
    labels = labels if labels is not None else sorted(set(gold))
    return float(f1_score(gold, pred, labels=labels, average="macro", zero_division=0))


def bootstrap_diff(gold, a, b, metric, n=10000, seed=0):
    """Difference metric(a) - metric(b), a 95% interval over resampled items, and a one-sided p (a > b)."""
    rng = np.random.default_rng(seed)
    gold, a, b = np.asarray(gold, dtype=object), np.asarray(a, dtype=object), np.asarray(b, dtype=object)
    base = metric(list(gold), list(a)) - metric(list(gold), list(b))
    diffs = []
    for _ in range(n):
        ix = rng.integers(0, len(gold), len(gold))
        diffs.append(metric(list(gold[ix]), list(a[ix])) - metric(list(gold[ix]), list(b[ix])))
    diffs = np.array(diffs)
    return {"diff": round(float(base), 4), "ci": [round(float(np.percentile(diffs, 2.5)), 4),
                                                  round(float(np.percentile(diffs, 97.5)), 4)],
            "p": float((np.sum(diffs <= 0) + 1) / (n + 1))}


def t_greater(x):
    from scipy import stats
    x = np.asarray(x, dtype=float)
    if np.allclose(x, x[0]):
        return 0.0 if x[0] > 0 else 1.0
    return float(stats.ttest_1samp(x, 0.0, alternative="greater").pvalue)


def holm(ps, alpha=0.05):
    """Holm step-down. Continues past a rejection in the wrong direction (that hypothesis is simply not
    supported); stops only at the first p that fails its threshold."""
    order = sorted(ps, key=lambda k: ps[k]["p"])
    out, m = {}, len(order)
    stopped = False
    for i, k in enumerate(order):
        thr = alpha / (m - i)
        ok = (not stopped) and ps[k]["p"] <= thr
        if not ok:
            stopped = True
        out[k] = {**ps[k], "threshold": round(thr, 5), "supported": bool(ok and ps[k].get("direction", True))}
    return out


# ------------------------------------------------------------------------------------------------ task A: acts
def clean_swda(t):
    t = re.sub(r"<[^>]*>", " ", t)                       # <laughter>, <noise>
    t = re.sub(r"\{[A-Z]\s*", " ", t).replace("}", " ")  # {F uh, } {D well} - keep the words
    t = re.sub(r"[\[\]+#/]|\(\(|\)\)|-/", " ", t)
    return re.sub(r"\s+", " ", t).strip(" ,")


def load_swda(path):
    """-> [(conversation, text, act)] from the swda.zip (or its unpacked folder)."""
    rows = []

    def take(fh):
        # an utterance interrupted by the other speaker carries on in a "+" row; the act belongs to the whole of
        # it, so each continuation is joined back onto that caller's last utterance
        utts, last = [], {}
        for r in csv.DictReader(io.TextIOWrapper(fh, encoding="utf-8", errors="replace")):
            tag = (r.get("act_tag") or "").strip()
            caller = r.get("caller")
            if tag == "+":
                if caller in last:
                    utts[last[caller]][1] += " " + (r.get("text") or "")
                continue
            base = re.split(r"[\^(,;]", tag)[0].strip()
            last[caller] = len(utts)
            utts.append([r.get("conversation_no"), r.get("text") or "", SWDA.get(base)])
        for conv, text, act in utts:
            text = clean_swda(text)
            if act and text:
                rows.append((conv, text, act))

    if os.path.isdir(path):
        for f in glob.glob(os.path.join(path, "**", "*.utt.csv"), recursive=True):
            if "__MACOSX" not in f:
                with open(f, "rb") as fh:
                    take(fh)
    else:
        with zipfile.ZipFile(path) as z:
            for name in z.namelist():
                if name.endswith(".utt.csv") and "__MACOSX" not in name and "/._" not in name:
                    with z.open(name) as fh:
                        take(fh)
    return rows


def split_by_conversation(rows, seed=0):
    convs = sorted({c for c, _, _ in rows})
    random.Random(seed).shuffle(convs)
    n = len(convs)
    part = {c: ("train" if i < 0.85 * n else "val" if i < 0.90 * n else "test") for i, c in enumerate(convs)}
    out = {"train": [], "val": [], "test": []}
    for c, t, a in rows:
        out[part[c]].append((t, a))
    return out


FILLER = re.compile(r"^(anyway|ok(ay)?|right|hmm+|so|well|alright|cool|nice|sure|i see|got it|wow|uh huh|yeah)\b[.!]*$",
                    re.I)


def rule_act(text):
    """nano's hand-written rules, as they stand, mapped onto the 12 acts."""
    from brain import discourse
    import language as L
    kind, _ = discourse.act(text)
    m = {"greet": "greet", "bye": "bye", "thanks": "thanks", "sorry": "sorry", "agree": "agree",
         "disagree": "disagree", "meaning": "meaning", "why": "question", "check": "question", "report": "question",
         "about-you": "question", "view": "question", "remember": "question", "follow-up": "question"}
    if kind in m:
        return m[kind]
    try:
        p = L.parse(text)
    except Exception:
        p = {"kind": "unknown"}
    if p.get("kind") in ("command", "instruct"):
        return "directive"
    if p.get("kind") == "ask" or text.strip().endswith("?") or re.match(
            r"^\W*(who|what|where|when|why|how|which|whose|is|are|do|does|did|can|could|will|would)\b", text, re.I):
        return "question"
    if FILLER.match(text.strip()):
        return "backchannel"
    return "statement"


def run_acts(a, net, voc, meta):
    t0 = time.time()
    rows = load_swda(a.swda)
    sp = split_by_conversation(rows, seed=0)
    nano = [json.loads(l) for l in open(a.nano_acts, encoding="utf-8") if l.strip()]
    print(f"switchboard: {len(rows)} utterances -> " + ", ".join(f"{k} {len(v)}" for k, v in sp.items()), flush=True)
    print("classes (train):", dict(collections.Counter(x for _, x in sp["train"]).most_common()), flush=True)
    if a.limit:
        for k in sp:
            sp[k] = sp[k][:a.limit if k == "train" else a.limit // 5]
    sets = {"train": sp["train"], "val": sp["val"], "test": sp["test"], "nano": [(r["text"], r["act"]) for r in nano]}
    ix = {c: i for i, c in enumerate(ACTS)}
    data = {}
    for name, items in sets.items():
        texts = [t for t, _ in items]
        X = cortex_codes(net, voc, texts)
        S = [sentence_features(t) for t in texts]
        y = torch.tensor([ix[c] for _, c in items])
        data[name] = {"texts": texts, "X": X, "S": S, "y": y, "gold": [c for _, c in items]}
        print(f"  coded {name}: {len(texts)} ({time.time() - t0:.0f}s)", flush=True)
    data["train"]["X"], stats = standardise(data["train"]["X"])
    for name in ("val", "test", "nano"):
        data[name]["X"], _ = standardise(data[name]["X"], stats)
    gain = balanced_gain(data["train"]["y"], len(ACTS))
    res = {"task": "acts", "grid": [GRID_LR, GRID_DECAY], "epochs": EPOCHS, "cortex": meta, "sizes": {k: len(v["gold"]) for k, v in data.items()}, "arms": {}}
    inputs = {"delta-words": (False, True), "delta-cortex": (True, False), "delta-both": (True, True)}
    keep = {}
    for arm, (dense, sparse) in inputs.items():
        pick = lambda d: (d["X"] if dense else None, d["S"] if sparse else None)
        grid = []
        for lr in GRID_LR:
            for dec in GRID_DECAY:
                m = Delta(data["train"]["X"].shape[1] if dense else 0, len(ACTS), lr, dec, gain, sparse, seed=0)
                m.fit(*pick(data["train"]), data["train"]["y"])
                pv = m.proba(*pick(data["val"])).argmax(1)
                grid.append((macro_f1(data["val"]["gold"], [ACTS[i] for i in pv], ACTS), lr, dec))
        best = max(grid)
        print(f"  {arm}: grid {[(round(g[0], 3), g[1], g[2]) for g in grid]} -> lr {best[1]} decay {best[2]}", flush=True)
        seeds, probs = [], {"test": 0, "nano": 0}
        for seed in range(SEEDS):
            m = Delta(data["train"]["X"].shape[1] if dense else 0, len(ACTS), best[1], best[2], gain, sparse,
                      seed=seed)
            m.fit(*pick(data["train"]), data["train"]["y"])
            one = {}
            for name in ("test", "nano"):
                p = m.proba(*pick(data[name]))
                probs[name] = probs[name] + p / SEEDS
                pred = [ACTS[i] for i in p.argmax(1)]
                one[name + "_f1"] = macro_f1(data[name]["gold"], pred)
                one[name + "_acc"] = float(np.mean([g == q for g, q in zip(data[name]["gold"], pred)]))
            seeds.append(one)
            if seed == 0 and arm == "delta-both":
                keep["model"] = m
        res["arms"][arm] = {"lr": best[1], "decay": best[2], "val_f1": round(best[0], 4), "seeds": seeds,
                            "test_f1": round(float(np.mean([s["test_f1"] for s in seeds])), 4),
                            "nano_f1": round(float(np.mean([s["nano_f1"] for s in seeds])), 4),
                            "nano_pred": [ACTS[i] for i in probs["nano"].argmax(1)],
                            "test_pred_avg": [ACTS[i] for i in probs["test"].argmax(1)]}
        print(f"  {arm}: test macro-F1 {res['arms'][arm]['test_f1']} nano {res['arms'][arm]['nano_f1']}", flush=True)
    # the rules being replaced, the tuned yardstick, and the floor
    for name in ("test", "nano"):
        res.setdefault("rules", {})[name + "_pred"] = [rule_act(t) for t in data[name]["texts"]]
        res["rules"][name + "_f1"] = round(macro_f1(data[name]["gold"], res["rules"][name + "_pred"]), 4)
    maj = collections.Counter(data["train"]["gold"]).most_common(1)[0][0]
    res["majority"] = {n + "_f1": round(macro_f1(data[n]["gold"], [maj] * len(data[n]["gold"])), 4)
                       for n in ("test", "nano")}
    res["tfidf-lr"] = tfidf_lr([t for t in data["train"]["texts"]], data["train"]["gold"],
                               data["val"]["texts"], data["val"]["gold"],
                               {n: data[n]["texts"] for n in ("test", "nano")})
    for n in ("test", "nano"):
        res["tfidf-lr"][n + "_f1"] = round(macro_f1(data[n]["gold"], res["tfidf-lr"][n + "_pred"]), 4)
    print("  rules:", {k: v for k, v in res["rules"].items() if k.endswith("f1")}, " tfidf-lr:",
          {k: v for k, v in res["tfidf-lr"].items() if k.endswith("f1") or k == "C"}, " majority:", res["majority"],
          flush=True)
    both = res["arms"]["delta-both"]
    h1 = bootstrap_diff(data["nano"]["gold"], both["nano_pred"], res["rules"]["nano_pred"], macro_f1)
    lr_test = res["tfidf-lr"]["test_f1"]
    h2 = {"diff": round(both["test_f1"] - lr_test, 4),
          "p": t_greater([s["test_f1"] - lr_test + 0.03 for s in both["seeds"]])}
    words = res["arms"]["delta-words"]["seeds"]
    h3 = {"diff": round(both["test_f1"] - res["arms"]["delta-words"]["test_f1"], 4),
          "p": t_greater([b["test_f1"] - w["test_f1"] for b, w in zip(both["seeds"], words)])}
    res["hypotheses"] = {"H1": {**h1, "claim": "delta-both beats rules on the nano set (macro-F1)"},
                         "H2": {**h2, "claim": "delta-both within 0.03 of tfidf-lr on Switchboard test"},
                         "H3": {**h3, "claim": "cortex adds: delta-both beats delta-words on Switchboard test"}}
    res["confusion_nano"] = [[t, g, p] for t, g, p in zip(data["nano"]["texts"], data["nano"]["gold"],
                                                            both["nano_pred"]) if g != p]
    m = keep["model"]
    torch.save({"task": "acts", "classes": ACTS, "readout": m.state(), "stats": stats, "dim": DIM,
                "cortex": meta, "lr": m.lr, "decay": m.decay}, os.path.join(a.out, "acts_readout.pt"))
    res["seconds"] = round(time.time() - t0)
    return res


def tfidf_lr(train_texts, train_y, val_texts, val_y, tests, analyzer="word"):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b|[?!]")
    Xtr = vec.fit_transform(train_texts)
    Xva = vec.transform(val_texts)
    best = None
    for C in GRID_C:
        lr = LogisticRegression(C=C, class_weight="balanced", max_iter=2000)
        lr.fit(Xtr, train_y)
        f = macro_f1(val_y, list(lr.predict(Xva)))
        if best is None or f > best[0]:
            best = (f, C, lr)
    out = {"C": best[1], "val_f1": round(best[0], 4)}
    for name, texts in tests.items():
        out[name + "_pred"] = list(best[2].predict(vec.transform(texts)))
    return out


# ------------------------------------------------------------------------------------------------ task B: facts
STOP = {"a", "an", "the", "my", "i", "to", "of", "and", "is", "am", "are", "have", "has", "in", "on", "at", "for",
        "with", "".strip()}


def detok(s):
    s = re.sub(r"\s+([.,!?;:%])", r"\1", str(s))
    s = re.sub(r"\s+'\s*(s|t|re|ve|ll|d|m)\b", r"'\1", s)
    return re.sub(r"\s+n't\b", "n't", s).strip()


def norm_words(s):
    out = []
    for w in O.words(str(s)):
        if w in STOP or not re.match(r"[a-z0-9]", w):
            continue
        out.append(w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w)
    return out


def load_dnli(path):
    """-> {split: [(sentence, (subject, relation, object))]}, unique sentences, test minus anything in train."""
    import pandas as pd
    files = {"train": "train", "dev": "dev", "test": "test"}
    out = {}
    for split, stem in files.items():
        f = glob.glob(os.path.join(path, "**", f"{stem}-*.parquet"), recursive=True)
        if not f:
            f = glob.glob(os.path.join(path, "**", f"{stem}*.parquet"), recursive=True)
        df = pd.read_parquet(f[0])
        seen = {}
        for s_col, t_col in (("sentence1", "triple1"), ("sentence2", "triple2")):
            for s, t in zip(df[s_col], df[t_col]):
                t = list(t) if t is not None else []
                if len(t) != 3 or not str(t[1]).strip() or str(t[1]) in ("<none>", "none"):
                    continue
                s = detok(s)
                if s and s not in seen:
                    seen[s] = (str(t[0]), str(t[1]), str(t[2]))
        out[split] = list(seen.items())
    train = {s for s, _ in out["train"]}
    out["test"] = [(s, t) for s, t in out["test"] if s not in train]
    out["dev"] = [(s, t) for s, t in out["dev"] if s not in train]
    return out


def object_labels(words, obj):
    target = set(norm_words(obj))
    lab = []
    for w in words:
        n = norm_words(w)
        lab.append(1 if n and n[0] in target else 0)
    return lab


def best_run(p):
    """The contiguous run of words most likely to be the object (sum of p - 0.5), or the single likeliest word."""
    if not len(p):
        return (0, 0)
    best, cur, start, span = -1e9, 0.0, 0, (int(np.argmax(p)), int(np.argmax(p)) + 1)
    for i, x in enumerate(p):
        if cur <= 0:
            cur, start = 0.0, i
        cur += x - 0.5
        if cur > best and x > 0.5:
            best, span = cur, (start, i + 1)
    return span


PETS = {"dog", "cat", "puppy", "kitten", "bird", "fish", "hamster", "horse", "rabbit", "turtle", "snake", "parrot",
        "pet", "lizard", "pig", "goldfish"}
FAMILY = {"brother", "sister", "mother", "father", "mom", "dad", "son", "daughter", "wife", "husband", "kid", "child",
          "children", "kids", "sibling", "grandma", "grandpa", "aunt", "uncle", "cousin", "twin"}
VEHICLES = {"car", "truck", "bike", "motorcycle", "bicycle", "van", "jeep", "boat", "tesla", "mustang"}


def rule_fact(sentence, rels):
    """nano's object grammar (brain/world.py), mapped onto Dialogue NLI relations where it has one."""
    from brain.world import World

    def rel(*keys):
        for k in keys:
            hit = [r for r in rels if k in r]
            if hit:
                return sorted(hit, key=len)[0]
        return None

    w = World()
    seen = w.observe(sentence)
    you = w.things["you"]
    for o in seen:
        if o["what"] in ("intro",) and o["thing"] in w.things:
            t = w.things[o["thing"]]
            k = t.kind or ""
            if k in PETS:
                return rel("have_pet", "pet"), k
            if k in FAMILY:
                return rel("have_family", "have_children", "family", "sibling"), k
            return rel("have", "own"), k
        if o["what"] == "has" and o["thing"] == "you":
            got = [w.things[h] for h in you.has if h in w.things]
            if got:
                k = got[-1].kind or ""
                if k in PETS:
                    return rel("have_pet", "pet"), k
                if k in FAMILY:
                    return rel("have_family", "have_children", "family"), k
                if k in VEHICLES:
                    return rel("have_vehicle", "vehicle", "own"), k
                return rel("own", "have"), k
        if o["what"] == "where" and o["thing"] == "you" and you.where in w.things:
            return rel("live_in_citystatecountry", "live_in"), w.things[you.where].name
        if o["what"] == "from" and o["thing"] == "you":
            return rel("place_origin", "origin", "live_in"), you.props.get("from", "")
        if o["what"] == "name":
            return rel("name"), you.props.get("name", "")
        if o["what"] == "kind" and o["thing"] == "you" and you.kinds and you.kinds[0] != "person":
            return rel("has_profession", "profession", "job"), you.kinds[0]
    return None, None


def run_facts(a, net, voc, meta):
    t0 = time.time()
    raw = load_dnli(a.dnli)
    if a.limit:
        raw = {k: v[:a.limit if k == "train" else a.limit // 5] for k, v in raw.items()}
    counts = collections.Counter(t[1] for _, t in raw["train"])
    rels = sorted(r for r, c in counts.items() if c >= 50) + ["other"]
    rix = {r: i for i, r in enumerate(rels)}
    relof = lambda r: r if r in rix else "other"
    print(f"dialogue nli: " + ", ".join(f"{k} {len(v)}" for k, v in raw.items()) + f"; {len(rels)} relations",
          flush=True)
    data = {}
    for split in ("train", "dev", "test"):
        items = raw[split]
        texts = [s for s, _ in items]
        X, tok = cortex_codes(net, voc, texts, tokens=True)
        words = [O.words(s)[:40] for s in texts]
        data[split] = {"texts": texts, "X": X, "tok": tok, "words": words,
                       "S": [sentence_features(s) for s in texts],
                       "y": torch.tensor([rix[relof(t[1])] for _, t in items]),
                       "rel": [relof(t[1]) for _, t in items], "obj": [t[2] for _, t in items],
                       "lab": [object_labels(w, t[2]) for w, (_, t) in zip(words, items)]}
        print(f"  coded {split}: {len(texts)} ({time.time() - t0:.0f}s)", flush=True)
    data["train"]["X"], stats = standardise(data["train"]["X"])
    for s in ("dev", "test"):
        data[s]["X"], _ = standardise(data[s]["X"], stats)
    gain = balanced_gain(data["train"]["y"], len(rels))

    # token-level data for the object readout
    def tokens_of(d, limit=None):
        Xs, Ss, ys, owner = [], [], [], []
        for i, (w, lab) in enumerate(zip(d["words"], d["lab"])):
            if not any(lab) and limit is not None:
                continue                 # training only on sentences whose object is visible in the words
            st = d["tok"][i]
            for j in range(min(len(w), len(st))):
                Xs.append(st[j].float())
                Ss.append(token_features(w, j))
                ys.append(lab[j])
                owner.append(i)
        return (torch.stack(Xs) if Xs else torch.zeros(0, net.H)), Ss, torch.tensor(ys), owner

    tt = {s: tokens_of(data[s], limit=True if s == "train" else None) for s in ("train", "dev", "test")}
    Xtok, tstats = standardise(tt["train"][0])
    tt["train"] = (Xtok, *tt["train"][1:])
    for s in ("dev", "test"):
        tt[s] = (standardise(tt[s][0], tstats)[0], *tt[s][1:])
    tgain = balanced_gain(tt["train"][2], 2)
    print(f"  object tokens: train {len(tt['train'][2])} ({int(tt['train'][2].sum())} object words)", flush=True)

    def triples(rel_pred, tok_p, d, owner):
        by = collections.defaultdict(list)
        for k, i in enumerate(owner):
            by[i].append(float(tok_p[k]))
        objs = []
        for i in range(len(d["texts"])):
            p = by.get(i, [])
            s, e = best_run(p) if p else (0, 0)
            objs.append(" ".join(d["words"][i][s:e]))
        exact = [r == g and norm_words(o) == norm_words(go) and bool(norm_words(go))
                 for r, g, o, go in zip(rel_pred, d["rel"], objs, d["obj"])]
        return objs, exact

    def token_f1(gold, pred):
        tp = sum(1 for g, p in zip(gold, pred) if g and p)
        fp = sum(1 for g, p in zip(gold, pred) if p and not g)
        fn = sum(1 for g, p in zip(gold, pred) if g and not p)
        return 2 * tp / max(1, 2 * tp + fp + fn)

    res = {"task": "facts", "grid": [GRID_LR, GRID_DECAY], "epochs": EPOCHS, "cortex": meta, "relations": rels, "sizes": {k: len(v["texts"]) for k, v in data.items()},
           "arms": {}}
    inputs = {"delta-words": (False, True), "delta-cortex": (True, False), "delta-both": (True, True)}
    keep = {}
    for arm, (dense, sparse) in inputs.items():
        pick = lambda d: (d["X"] if dense else None, d["S"] if sparse else None)
        tpick = lambda t: (t[0] if dense else None, t[1] if sparse else None)
        grid = []
        for lr in GRID_LR:
            for dec in GRID_DECAY:
                m = Delta(data["train"]["X"].shape[1] if dense else 0, len(rels), lr, dec, gain, sparse, 0)
                m.fit(*pick(data["train"]), data["train"]["y"])
                tm = Delta(net.H if dense else 0, 2, lr, dec, tgain, sparse, 0)
                tm.fit(*tpick(tt["train"]), tt["train"][2])
                rp = [rels[i] for i in m.proba(*pick(data["dev"])).argmax(1)]
                tp = tm.proba(*tpick(tt["dev"]))[:, 1]
                _, ex = triples(rp, tp, data["dev"], tt["dev"][3])
                grid.append((float(np.mean(ex)), lr, dec))
        best = max(grid)
        print(f"  {arm}: grid {[(round(g[0], 3), g[1], g[2]) for g in grid]} -> lr {best[1]} decay {best[2]}",
              flush=True)
        seeds, rp_avg, tp_avg = [], 0, 0
        for seed in range(SEEDS):
            m = Delta(data["train"]["X"].shape[1] if dense else 0, len(rels), best[1], best[2], gain, sparse, seed)
            m.fit(*pick(data["train"]), data["train"]["y"])
            tm = Delta(net.H if dense else 0, 2, best[1], best[2], tgain, sparse, seed)
            tm.fit(*tpick(tt["train"]), tt["train"][2])
            pr = m.proba(*pick(data["test"]))
            tp = tm.proba(*tpick(tt["test"]))[:, 1]
            rp_avg, tp_avg = rp_avg + pr / SEEDS, tp_avg + tp / SEEDS
            rpred = [rels[i] for i in pr.argmax(1)]
            _, ex = triples(rpred, tp, data["test"], tt["test"][3])
            seeds.append({"exact": float(np.mean(ex)), "rel_f1": macro_f1(data["test"]["rel"], rpred),
                          "rel_acc": float(np.mean([r == g for r, g in zip(rpred, data["test"]["rel"])])),
                          "tok_f1": token_f1(tt["test"][2].tolist(), (tp > 0.5).tolist())})
            if seed == 0 and arm == "delta-both":
                keep["rel"], keep["obj"] = m, tm
        rpred = [rels[i] for i in rp_avg.argmax(1)]
        objs, ex = triples(rpred, tp_avg, data["test"], tt["test"][3])
        res["arms"][arm] = {"lr": best[1], "decay": best[2], "dev_exact": round(best[0], 4), "seeds": seeds,
                            **{k: round(float(np.mean([s[k] for s in seeds])), 4) for k in seeds[0]},
                            "exact_avg_items": [bool(x) for x in ex]}
        print(f"  {arm}: " + ", ".join(f"{k} {res['arms'][arm][k]}" for k in seeds[0]), flush=True)
        if arm == "delta-both":
            res["examples"] = [[s, r, o, gr, go] for s, r, o, gr, go in
                               zip(data["test"]["texts"][:40], rpred[:40], objs[:40], data["test"]["rel"][:40],
                                   data["test"]["obj"][:40])]
    # the object grammar it would replace
    rule_items = []
    for s, rel, obj in zip(data["test"]["texts"], data["test"]["rel"], data["test"]["obj"]):
        r, o = rule_fact(s, rels)
        rule_items.append(bool(r) and relof(r) == rel and norm_words(o or "") == norm_words(obj) and bool(norm_words(obj)))
    res["rules"] = {"exact": round(float(np.mean(rule_items)), 4), "exact_items": rule_items,
                    "covered": round(float(np.mean([rule_fact(s, rels)[0] is not None
                                                    for s in data["test"]["texts"][:3000]])), 4)}
    # the tuned yardstick: logistic regression for the relation and for each word
    lr_rel = tfidf_lr(data["train"]["texts"], data["train"]["rel"], data["dev"]["texts"], data["dev"]["rel"],
                      {"test": data["test"]["texts"]})
    lr_tok = token_lr(tt, rels)
    _, ex = triples(lr_rel["test_pred"], lr_tok, data["test"], tt["test"][3])
    res["tfidf-lr"] = {"C": lr_rel["C"], "rel_f1": round(macro_f1(data["test"]["rel"], lr_rel["test_pred"]), 4),
                       "exact": round(float(np.mean(ex)), 4),
                       "tok_f1": round(token_f1(tt["test"][2].tolist(), (lr_tok > 0.5).tolist()), 4)}
    print("  rules:", {k: v for k, v in res["rules"].items() if not k.endswith("items")}, " tfidf-lr:",
          res["tfidf-lr"], flush=True)
    both = res["arms"]["delta-both"]
    a_ok, b_ok = np.array(both["exact_avg_items"], dtype=float), np.array(rule_items, dtype=float)
    rng = np.random.default_rng(0)
    boots = np.array([(a_ok[ix] - b_ok[ix]).mean() for ix in (rng.integers(0, len(a_ok), len(a_ok))
                                                               for _ in range(10000))])
    h4 = {"diff": round(float((a_ok - b_ok).mean()), 4),
          "ci": [round(float(np.percentile(boots, 2.5)), 4), round(float(np.percentile(boots, 97.5)), 4)],
          "p": float((np.sum(boots <= 0) + 1) / (len(boots) + 1))}
    h5 = {"diff": round(both["rel_f1"] - res["tfidf-lr"]["rel_f1"], 4),
          "p": t_greater([s["rel_f1"] - res["tfidf-lr"]["rel_f1"] + 0.03 for s in both["seeds"]])}
    h6 = {"diff": round(both["tok_f1"] - res["arms"]["delta-words"]["tok_f1"], 4),
          "p": t_greater([b["tok_f1"] - w["tok_f1"] for b, w in zip(both["seeds"],
                                                                     res["arms"]["delta-words"]["seeds"])])}
    res["hypotheses"] = {"H4": {**h4, "claim": "delta-both beats the object grammar on triple exact match"},
                         "H5": {**h5, "claim": "delta-both relation macro-F1 within 0.03 of tfidf-lr"},
                         "H6": {**h6, "claim": "cortex helps find the object: delta-both beats delta-words (token F1)"}}
    for k in ("exact_avg_items",):
        for arm in res["arms"].values():
            arm.pop(k, None)
    res["rules"].pop("exact_items", None)
    torch.save({"task": "facts", "relations": rels, "rel_readout": keep["rel"].state(),
                "obj_readout": keep["obj"].state(), "stats": stats, "token_stats": tstats, "dim": DIM,
                "cortex": meta}, os.path.join(a.out, "facts_readout.pt"))
    res["seconds"] = round(time.time() - t0)
    return res


def token_lr(tt, rels):
    """Logistic regression over the same sparse word-in-place code, C tuned on dev token F1."""
    from scipy import sparse
    from sklearn.linear_model import LogisticRegression

    def mat(S):
        rows = np.repeat(np.arange(len(S)), [len(s) for s in S])
        cols = np.array([i for s in S for i in s])
        return sparse.csr_matrix((np.ones(len(cols)), (rows, cols)), shape=(len(S), DIM))

    Xtr, Xdv, Xte = mat(tt["train"][1]), mat(tt["dev"][1]), mat(tt["test"][1])
    best = None
    for C in GRID_C:
        m = LogisticRegression(C=C, class_weight="balanced", max_iter=1000)
        m.fit(Xtr, tt["train"][2].numpy())
        p = m.predict_proba(Xdv)[:, 1]
        g = tt["dev"][2].numpy()
        tp, fp, fn = ((p > .5) & (g == 1)).sum(), ((p > .5) & (g == 0)).sum(), ((p <= .5) & (g == 1)).sum()
        f = 2 * tp / max(1, 2 * tp + fp + fn)
        if best is None or f > best[0]:
            best = (f, C, m)
    return best[2].predict_proba(Xte)[:, 1]


# ------------------------------------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", choices=["acts", "facts", "both"], default="both")
    ap.add_argument("--swda")
    ap.add_argument("--dnli")
    ap.add_argument("--cortex", required=True)
    ap.add_argument("--nano-acts", default=os.path.join(ROOT, "tests", "nano_acts.jsonl"))
    ap.add_argument("--out", default=".")
    ap.add_argument("--limit", type=int, default=0, help="pilot: cap the training set")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    torch.set_num_threads(max(1, os.cpu_count() or 1))
    net, voc, meta = load_cortex(a.cortex)
    print("cortex:", meta, flush=True)
    out = {}
    if a.task in ("acts", "both"):
        out["acts"] = run_acts(a, net, voc, meta)
    if a.task in ("facts", "both"):
        out["facts"] = run_facts(a, net, voc, meta)
    ps = {}
    for task in out.values():
        for h, v in task["hypotheses"].items():
            ps[h] = {"p": v["p"], "diff": v["diff"], "claim": v["claim"], "direction": v["diff"] > -0.03
                     if h in ("H2", "H5") else v["diff"] > 0}
    out["verdict"] = holm(ps)
    json.dump(out, open(os.path.join(a.out, "results.json"), "w"), indent=1, default=str)
    print(json.dumps(out["verdict"], indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/brain/brain/discourse.py
"""Discourse: the part of talking that is not about the world.

A conversation is not a sequence of independent questions. People drop most of a sentence when the rest is
obvious ("and Bob?"), ask for grounds ("why?"), check a claim ("are you sure?"), correct what was just said
("no, Rex is older"), and expect the other side to occasionally want something. None of that is about relations
between things, and none of it was possible while every turn was parsed from scratch.

What makes it work here is the task set - prefrontal cortex holding not just *what* is being discussed but *what
is being done with it*. "And Bob?" is answerable only because the question form from the previous turn is still
held: swap one argument, keep the rest. That is the same held representation that lets a person keep applying a
rule to new items without being told the rule again.

Initiative comes from the map's own shape rather than from a script: two groups of facts that nothing links are
exactly where a question would help most, so that is what it asks about.
"""
import re

FOLLOW_UP = (re.compile(r"^(?:and|what about|how about|ok(?:ay)?,? and)\s+(.+?)\s*\??$", re.I),)
WHY = re.compile(r"^(why|why is that|how do you know|how come|says who|on what grounds)\b", re.I)
# "what do you mean?" is about the last thing said, never about the world - searching it found a pop song
MEANING = re.compile(r"^(?:(?:ok(?:ay)?|sorry|um+|uh+|wait|hm+|so|but)[,.!]?\s+)*"
                     r"(what do you mean|what does that mean|what did you mean|what\?*$|huh|come again|pardon|"
                     r"i don'?t (understand|get it|get that|follow)|what are you (talking about|saying|on about)|"
                     r"(can you |could you )?(explain|clarify|rephrase) (that|this|yourself|what you mean|what you said)|"
                     r"say (that|it) again|meaning\?*$|what\?+|that makes no sense|that doesn'?t make sense|"
                     r"what is that supposed to mean|what'?s that supposed to mean)", re.I)
SURE = re.compile(r"^(are you sure|really|is that right|you sure|certain\??|"
                  r"how (sure|confident|certain) (are you|is that)|how do you rate)\b", re.I)
AGREE = re.compile(r"^(yes|yeah|yep|right|correct|exactly|that's right|true)\b", re.I)
# "actually no, Ollie is older than Rex" is a correction with a softener in front of it. Missing the softener
# leaves the refusal attached to the fact and files a thing called "no, Ollie", which is worse than missing
# the correction altogether.
SOFTENER = r"(?:actually|wait|hang on|sorry|hmm|hold on|oh)?[,\s]*"
DISAGREE = re.compile(r"^" + SOFTENER + r"(no|nope|wrong|that's wrong|not true|incorrect|that's not right)\b",
                      re.I)
THANKS = re.compile(r"^(thanks|thank you|cheers|ta)\b", re.I)
SORRY = re.compile(r"^(sorry|my bad|apologies)\b", re.I)
GREET = re.compile(r"^(hi|hey|hello|good (morning|evening|afternoon)|yo|howdy)\b", re.I)
BYE = re.compile(r"^(bye|goodbye|see you|good night|later)\b", re.I)
SELF_REPORT = re.compile(r"^(how (was|is|has) your (day|week|time)|how('s| is| are) (it going|things|you)|"
                         r"what have you (been up to|been doing|done|learned|learnt)|what's new|"
                         r"how are you doing|tell me about your (day|self)|"
                         r"what (did|have) you (do|been doing|done|get up to)|"
                         r"(was|were) (any of )?(it|that|this|they) (hard|difficult|tricky|easy|interesting)|"
                         r"what (was|has been) (hard|difficult|tricky|the hard part)|"
                         r"how (did|has) (it|that|today) (go|gone|been)|what was (it|that) like)", re.I)
ABOUT_YOU = re.compile(r"^(who|what) are you|^how do you work|^what can you do|"
                       r"^(who|what) am i\b|^what do you know about me|^are you (an? )?(ai|llm|robot|human)|"
                       r"\b(you|your)\b.*\b(think|feel|like|want|remember|made)\b|"
                       r"^(what|when) (have|did) you got? .*wrong|^what did you get wrong|"
                       r"\byour (name|favou?rite|age|job|purpose|creator|maker|opinion|feelings?)\b|"
                       r"\byour name\b|\bname (do|did|should|would) (you|i)\b|\b(are|were) you called\b|\bcall (you|yourself)\b|"
                       r"\b(did|didn'?t)( not)? you (not |just )?(say|said|answer|respond|reply)\b", re.I)


VIEW = re.compile(r"^(what do you (make of|think of|think about|reckon)|"
                  r"what('s| is) your (view|take|opinion|sense|impression)|"
                  r"how do you (feel|rate|see) |what do you think (of|about) |"
                  r"which (one )?(should|would)|what (would|should) (i|we) |what else (would|could|do you) |"
                  r"(what|anything) else (would|could) help|how would you|do you have (a view|thoughts|an opinion))",
                  re.I)
REMEMBER = re.compile(r"^(do you |can you )?remember (what|when|that|anything)?\s*(i|you|we)?\s*(said|told)?"
                      r"\s*(about)?\s*(.+)?$", re.I)


def act(text):
    """The dialogue act of an utterance, before anything is looked up. None means 'about the world'."""
    t = text.strip()
    for pattern in FOLLOW_UP:
        m = pattern.match(t)
        if m and len(m.group(1).split()) <= 4:
            return "follow-up", m.group(1).strip(" ?.")
    if VIEW.match(t):
        return "view", None
    m = REMEMBER.match(t)
    if m and m.group(6):
        return "remember", m.group(6).strip(" ?.")
    if SELF_REPORT.match(t):
        return "report", None
    if MEANING.match(t):
        return "meaning", None
    if WHY.match(t):
        return "why", None
    if SURE.match(t):
        return "check", None
    if THANKS.match(t):
        return "thanks", None
    if SORRY.match(t):
        return "sorry", None
    if GREET.match(t):
        return "greet", None
    if BYE.match(t):
        return "bye", None
    if DISAGREE.match(t):
        return "disagree", t
    if AGREE.match(t):
        return "agree", None
    if ABOUT_YOU.search(t):
        return "about-you", None
    return None, None


class TaskSet:
    """What is being done, held across turns so an incomplete sentence can borrow it.

    Prefrontal cortex holds a task rule, not just a topic: given "is X older than Y", the rule is "compare by
    age", and it survives until something replaces it. Holding it is what makes "and Bob?" a question at all.
    """

    def __init__(self):
        self.question = None      # the last full question, as parsed
        self.answer = None        # what was said back, with its grounds
        self.offered = 0          # the turn the brain last asked something of its own

    def hold(self, parsed, answer=None):
        if parsed.get("kind") == "ask":
            self.question = dict(parsed)
        if answer is not None:
            self.answer = answer

    def fill(self, term):
        """'and Bob?' -> the held question with one argument swapped. None if there is no rule to apply."""
        if not self.question:
            return None
        q = dict(self.question)
        key = (term.lower().strip(), term.strip())
        # the new term replaces whichever argument was *not* the one the question was mainly about: for
        # "is Alice older than Bob", "and Cara?" means Alice against Cara
        if q.get("b"):
            q["b"] = key
        elif q.get("a"):
            q["a"] = key
        else:
            q["a"] = key
        return q


def curiosity(maps, task, turn, every=4):
    """Something worth asking about, or None. The map's own gaps decide: two groups of facts with nothing
    between them cannot be compared, and one fact would join them."""
    if turn - task.offered < every:
        return None
    for space, m in maps.maps.items():
        if space in ("kind", "involved", "origin"):
            continue          # categories and associations are not scales; "compare by kind" means nothing
        groups = {}
        for n in m.names:
            if m.knows(n):
                groups.setdefault(m.island[m.index[n]], []).append(n)
        if len(groups) > 1:
            (_, first), (_, second) = sorted(groups.items())[:2]
            task.offered = turn
            dim = "where" if space == "place" else space
            if space == "place":
                return f"Where is {second[0]} compared with {first[0]}? Nothing you've told me links them."
            return f"How do {first[0]} and {second[0]} compare by {dim}? I can't place them on the same scale yet."
    return None

In [ ]:
%%writefile /kaggle/working/brain/brain/world.py
"""A sense of objects, of self, and of the world they are in.

The maps hold *relations* - Rex is older than Max, the cafe is north of the park - and that is the hippocampal
half of knowing a world: where things sit relative to each other. What they lacked is the other half: that Rex
is a thing at all. Told "my dog Rex is older than Max", the map placed a name on an age scale and dropped that
Rex is a dog, and that he is yours. Asked "what is Rex?" it could only say where he sits.

Object files (Kahneman & Treisman; perirhinal and ventral-stream cortex in the brain) are what keep a thing the
same thing across mentions: one record per object, and every property, owner, possession and whereabouts bound
to it as it is heard. Three objects are always here:

  you     the person talking - who they are, where they live, what is theirs, how they said they feel
  nano    itself, as an object in its own world: what it is, where it runs, what it holds. The sense of self is
          not a description it was given; it is this object plus interoception - reading its own state (how
          much it holds unconsolidated, how often it has been lost or corrected lately) the way a body reads
          hunger and fatigue
  things  everything else anyone mentioned, or that it read about

The world is all of them together, with what connects them. Everything here carries where it came from - told
or read - because a world model that forgets its sources is a confabulation waiting to happen.
"""
import re

COLOURS = {"red", "orange", "yellow", "green", "blue", "purple", "pink", "brown", "black", "white", "grey", "gray",
           "golden", "silver", "beige", "tan", "cream", "violet", "teal", "navy", "spotted", "striped"}
SIZES = {"big", "small", "tiny", "huge", "large", "little", "tall", "short", "fat", "thin", "heavy", "light",
         "long", "giant", "enormous"}
FEELINGS = {"tired", "sad", "happy", "bored", "angry", "stressed", "sick", "hungry", "excited", "lonely", "fine",
            "good", "great", "okay", "ok", "awful", "anxious", "nervous", "busy", "sleepy", "confused", "upset"}
ANIMATE = {"dog", "cat", "person", "man", "woman", "boy", "girl", "friend", "brother", "sister", "mother", "father",
           "mom", "dad", "son", "daughter", "wife", "husband", "boss", "teacher", "student", "puppy", "kitten",
           "bird", "horse", "rabbit", "hamster", "fish", "pet", "child", "kid", "baby", "cousin", "aunt", "uncle",
           "grandma", "grandpa", "roommate", "partner", "girlfriend", "boyfriend", "neighbor", "neighbour"}
PRONOUN = {"he", "she", "it", "they", "this", "that", "there", "these", "those", "what", "who", "which", "here",
           "we", "him", "her", "them", "everyone", "everything", "someone", "something", "nothing", "nobody"}
NOT_TRAIT = {"here", "there", "home", "back", "done", "gone", "out", "in", "up", "down", "right", "wrong", "true",
             "false", "sure", "not", "late", "early", "better", "worse", "older", "younger", "bigger", "smaller",
             "taller", "shorter", "next", "same", "different", "possible", "impossible"}
QUESTION = re.compile(r"^\W*(who|what|whats|what's|where|when|why|how|which|whose|is|are|am|do|does|did|can|could|"
                      r"will|would|should|have|has)\b", re.I)
DET = r"(?:a|an|one|two|three|four|five|some|\d+)"


class Thing:
    def __init__(self, key, name, turn=0, source="told"):
        self.key, self.name = key, name
        self.kinds = []            # what it is, most recent first ("dog")
        self.owner = None          # key of whose it is
        self.props = {}            # attribute -> value ("colour" -> "brown")
        self.has = []              # keys of things it has
        self.where = None          # key of where it is / lives
        self.at = "in"             # how: in Dallas, at UCLA, near the park
        self.aliases = set()
        self.source = source       # told | read | self
        self.first = self.last = turn
        self.notes = []            # the sentences it was built from, with their source

    @property
    def kind(self):
        return self.kinds[0] if self.kinds else None

    def is_kind(self, kind):
        k = kind.rstrip("s")
        return any(x == kind or x == k or x.rstrip("s") == k for x in self.kinds)


class World:
    YOU, ME = "you", "nano"

    def __init__(self):
        self.things = {}
        self.turn = 0
        you = self._make(self.YOU, "you", source="self")
        you.kinds = ["person"]
        me = self._make(self.ME, "nano", source="self")
        me.kinds = ["brain"]
        home = self._make("home", "a computer", source="self")
        me.where = home.key
        self.feelings = []        # how the person said they felt, with the turn

    def set_home(self, where):
        self.things["home"].name = where

    # ------------------------------------------------------------------ objects
    def _make(self, key, name, source="told"):
        t = self.things.get(key)
        if t is None:
            t = Thing(key, name, self.turn, source)
            self.things[key] = t
        return t

    def find(self, np):
        """The thing a noun phrase refers to, if there is one - never creates."""
        return self._resolve(np, create=False)

    def _resolve(self, np, create=True, source="told"):
        np = re.sub(r"\s+", " ", str(np or "")).strip(" ,!?\"'")
        np = re.sub(r"(?<!\b[A-Z])\.+$", "", np)          # "U.S." keeps its point, "UCLA." does not
        low = np.lower()
        if not low or low in PRONOUN:
            return None
        if low in ("i", "me", "myself"):
            return self.things[self.YOU]
        if low in ("you", "yourself", "nano"):
            return self.things[self.ME]
        m = re.match(r"^(my|our|your)\s+(.+)$", np, re.I)
        if m:
            owner = self.YOU if m.group(1).lower() in ("my", "our") else self.ME
            part = m.group(2).strip()
            # "my dog" is the dog of mine already known, if there is exactly one
            mine = [t for t in self.things.values() if t.owner == owner and t.is_kind(part.lower())]
            if len(mine) == 1:
                return mine[0]
            if mine:
                return mine[-1]
            if not create:
                return None
            key = ("your " if owner == self.YOU else "my ") + part.lower()
            t = self._make(key, key, source)
            t.owner = owner
            t.kinds.insert(0, part.split()[-1].lower())
            return t
        m = re.match(r"^(?:the|a|an)\s+(.+)$", np, re.I)
        if m:
            np, low = ("the " + m.group(1) if np.lower().startswith("the ") else m.group(1)), m.group(1).lower()
        for t in self.things.values():
            if t.key == low or low in t.aliases:
                return t
        if not create:
            return None
        return self._make(low, np, source)

    def mention(self, key, name, source="told"):
        """A thing came up somewhere else (placed on a map, read about): make sure it exists as an object."""
        if not key or key.lower() in PRONOUN:
            return None
        t = self._make(key.lower(), name, source)
        t.last = self.turn
        return t

    def read_lead(self, title, sentence):
        """A page's first sentence says what its subject is and, often, where: 'Dallas is a city in the U.S.
        state of Texas...' -> an object 'Dallas', a city, in the U.S. Filed as read, never as told."""
        m = re.search(r"\b(?:is|was) (?:a|an|the) ((?:[a-z-]+ ){0,3}?)([a-z-]+)\b"
                      r"(?:,? (?:located |situated |based )?(?:in|of) (?:the )?((?:[A-Z][\w.'-]*\s?){1,4}))?",
                      sentence or "")
        if not m or not title:
            return None
        t = self._resolve(title, source="read")
        if t is None:
            return None
        kind = m.group(2).lower()
        if kind not in t.kinds and kind not in ("one", "part", "member", "kind", "type", "form"):
            t.kinds.append(kind)
        if m.group(3) and not t.where:
            p = self._resolve(m.group(3).strip(), source="read")
            if p is not None and p is not t:
                t.where = p.key
                if not p.kinds:
                    p.kinds.append("place")
        t.last = self.turn
        return t

    # ------------------------------------------------------------------ hearing
    def observe(self, sentence, source="told", apply=True):
        """What a sentence says about objects -> [{"what", "say"}], applied to the world as it is heard.

        A question says nothing about the world, so nothing is taken from one. Possessive introductions are
        noticed anywhere in a sentence ("my dog Rex is older than Max" is also news that Rex is a dog, and
        yours); whole-sentence statements - properties, possessions, whereabouts, owners - only when the
        sentence is one."""
        s = re.sub(r"(?<!\b[A-Z])[.!]+$", "", re.sub(r"\s+", " ", sentence.strip()))   # keeps "the U.S."
        if not s or s.endswith("?") or QUESTION.match(s):
            return []
        s = re.sub(r"\b(I)'m\b", r"\1 am", s, flags=re.I)
        out = []
        self.turn += 1 if apply else 0
        # possessive introductions, anywhere: "my dog Rex", "Rex, my dog"
        for m in re.finditer(r"\b(?:[Mm]y|[Oo]ur) ([a-z]+) (?:named |called )?([A-Z][\w'-]+)\b", s):
            out += self._intro(m.group(2), m.group(1), self.YOU, source)
        for m in re.finditer(r"\b([A-Z][\w'-]+), (?:[Mm]y|[Oo]ur) ([a-z]+)\b", s):
            out += self._intro(m.group(1), m.group(2), self.YOU, source)
        s = re.sub(r", (?:[Mm]y|[Oo]ur) [a-z]+,", "", s)       # the apposition is taken; the claim is about Luna
        for rule in (self._named_intro, self._name, self._is_mine, self._belongs, self._lives, self._from,
                     self._has, self._is_a, self._trait):
            got = rule(s, source)
            if got:
                out += got
                break
        return out

    def _intro(self, name, kind, owner, source):
        t = self._resolve(name, source=source)
        if t is None:
            return []
        new_kind = kind.lower() not in t.kinds
        if new_kind:
            t.kinds.insert(0, kind.lower())
        was = t.owner
        t.owner = owner
        t.last = self.turn
        if not new_kind and was == owner:
            return []
        return [{"what": "intro", "thing": t.key, "say": f"{t.name} is {self.whose(t)} {kind.lower()}"}]

    def _named_intro(self, s, source):
        m = re.match(rf"^(?:i|we) (?:have|own|got|'ve got|have got|ve got) (?:{DET} )?((?:[a-z]+ )?[a-z]+?)s? "
                     rf"(?:named|called) (.+)$", s, re.I)
        if not m:
            return []
        kind = m.group(1).split()[-1]
        names = [n.strip() for n in re.split(r",\s*|\s+and\s+", m.group(2)) if n.strip()]
        out = []
        for n in names:
            out += self._intro(n, kind, self.YOU, source)
        return out

    def _name(self, s, source):
        m = re.match(r"^(?:my name is|i'm called|i am called|call me|i am|i'm) ([A-Z][\w'-]+)$", s, re.I)
        if not m or (m.group(1).lower() in FEELINGS | SIZES | NOT_TRAIT) or not m.group(1)[:1].isupper():
            return []
        you = self.things[self.YOU]
        you.props["name"] = m.group(1)
        you.aliases.add(m.group(1).lower())
        return [{"what": "name", "thing": "you", "say": f"your name is {m.group(1)}"}]

    def _is_mine(self, s, source):
        m = re.match(r"^(.+?) (?:is|are) (my|our|your|[A-Z][\w'-]*'s) ([a-z]+)$", s)
        if not m:
            return []
        who = m.group(2).lower()
        owner = (self.YOU if who in ("my", "our") else self.ME if who == "your" else
                 self._resolve(m.group(2)[:-2], source=source).key)
        return self._intro(m.group(1), m.group(3), owner, source)

    def _belongs(self, s, source):
        m = re.match(r"^(.+?) belongs? to (.+)$", s, re.I)
        if not m:
            return []
        t, o = self._resolve(m.group(1), source=source), self._resolve(m.group(2), source=source)
        if not t or not o:
            return []
        t.owner = o.key
        return [{"what": "owner", "thing": t.key, "say": f"{t.name} is {self.whose(t, bare=True)}"}]

    def _lives(self, s, source):
        m = re.match(r"^(.+?) (?:lives?|stays?|is based|works?|sleeps?|am|is|are|'m|'s) (in|at|near|on) (.+)$", s,
                     re.I)
        if not m or m.group(1).lower() in PRONOUN or re.match(r"^(love|trouble|charge|fact|time|a hurry)\b",
                                                                  m.group(3), re.I):
            return []
        t, p = self._resolve(m.group(1), source=source), self._resolve(m.group(3), source=source)
        if not t or not p or t is p:
            return []
        t.where, t.at = p.key, m.group(2).lower()
        if not p.kinds:
            p.kinds.append("place")
        verb = "live" if t.key == self.YOU else "lives" if (t.is_kind("person") or any(
            t.is_kind(k) for k in ANIMATE)) else "is"
        return [{"what": "where", "thing": t.key,
                 "say": f"{'you' if t.key == self.YOU else t.name} {verb} {m.group(2).lower()} {p.name}"}]

    def _from(self, s, source):
        m = re.match(r"^(.+?) (?:is|am|are|'m|comes?|come) from (.+)$", s, re.I)
        if not m or m.group(1).lower() in PRONOUN:
            return []
        t, p = self._resolve(m.group(1), source=source), self._resolve(m.group(2), source=source)
        if not t or not p:
            return []
        t.props["from"] = p.name
        if not p.kinds:
            p.kinds.append("place")
        return [{"what": "from", "thing": t.key, "say": f"{'you are' if t.key == self.YOU else t.name + ' is'} "
                                                         f"from {p.name}"}]

    def _has(self, s, source):
        m = re.match(rf"^(.+?) (?:has|have|owns|own|has got|have got|'ve got|ve got) ({DET}|the) (.+)$", s, re.I)
        if not m or m.group(1).lower() in PRONOUN:
            return []
        t = self._resolve(m.group(1), source=source)
        if not t:
            return []
        words = m.group(3).split()
        if len(words) > 4:
            return []
        noun, adjs = words[-1].lower(), [w.lower() for w in words[:-1]]
        many = m.group(2).lower() not in ("a", "an", "one", "the")
        noun1 = noun[:-1] if many and noun.endswith("s") else noun
        key = (("your " if t.key == self.YOU else "my " if t.key == self.ME else t.key + "'s ") +
               " ".join(adjs + [noun1]))
        thing = self._make(key, key, source)
        if t.key not in (self.YOU, self.ME):
            thing.name = t.name + "'s " + " ".join(adjs + [noun1])       # "Rex's red ball", as he is written
        thing.kinds = [noun1]
        thing.owner = t.key
        for a in adjs:
            self._set_trait(thing, a)
        if thing.key not in t.has:
            t.has.append(thing.key)
        who = "you have" if t.key == self.YOU else f"{t.name} has"
        return [{"what": "has", "thing": t.key, "say": f"{who} {m.group(2).lower()} {' '.join(words).lower()}"}]

    def _is_a(self, s, source):
        m = re.match(r"^(.+?) (?:is|am|are|'m|'s) (?:a|an|the) ((?:(?!(?:in|from|near|on|at|of) )[a-z-]+ ){0,2}[a-z-]+?)(?:s)?"
                     r"(?: (?:in|from|near|on|at|of) (.+))?$", s, re.I)
        if not m or m.group(1).lower() in PRONOUN:
            return []
        t = self._resolve(m.group(1), source=source)
        if not t:
            return []
        words = m.group(2).lower().split()
        kind, adjs = words[-1], words[:-1]
        if kind in ("kind", "type", "sort", "lot", "bit"):
            return []
        if kind not in t.kinds:
            t.kinds.insert(0, kind)
        for a in adjs:
            self._set_trait(t, a)
        said = f"{'you are' if t.key == self.YOU else t.name + ' is'} a{'n' if (adjs or [kind])[0][0] in 'aeiou' else ''} " \
               f"{' '.join(words)}"
        if m.group(3):
            p = self._resolve(m.group(3), source=source)
            if p:
                prep = re.search(r" (in|from|near|on|at|of) ", s)
                prep = prep.group(1).lower() if prep else "in"
                if prep in ("at", "of"):
                    # "a student at UCLA" says where they study, not where they live
                    t.props[kind + " " + prep] = p.name
                else:
                    t.where, t.at = p.key, prep
                if not p.kinds:
                    p.kinds.append("place")
                said += f" {prep} {p.name}"
        return [{"what": "kind", "thing": t.key, "say": said}]

    def _trait(self, s, source):
        m = re.match(r"^(.+?) (?:is|am|are|'m|'s|looks?|feels?|seems?|was|were) (?:very |really |so |quite |pretty |"
                     r"kind of |a bit |super )?([a-z]+)$", s, re.I)
        if not m or m.group(1).lower() in PRONOUN or m.group(2).lower() in NOT_TRAIT:
            return []
        word = m.group(2).lower()
        if word.endswith("er") and word not in COLOURS | FEELINGS | {"clever", "silver", "sober", "tender"}:
            return []                   # a comparative with its "than" dropped belongs to the map
        t = self._resolve(m.group(1), source=source)
        if not t:
            return []
        attr = self._set_trait(t, word)
        if t.key == self.YOU and attr == "feeling":
            self.feelings.append((self.turn, word))
            return [{"what": "feeling", "thing": "you", "say": f"you're {word}"}]
        return [{"what": "trait", "thing": t.key,
                 "say": f"{'you are' if t.key == self.YOU else t.name + ' is'} {word}"}]

    def _set_trait(self, t, word):
        attr = ("colour" if word in COLOURS else "size" if word in SIZES else
                "feeling" if word in FEELINGS and (t.key == self.YOU or t.is_kind("person")) else "trait")
        if attr == "trait":
            traits = t.props.setdefault("traits", [])
            if word not in traits:
                traits.append(word)
        else:
            t.props[attr] = word
        return attr

    # ------------------------------------------------------------------ saying
    def whose(self, t, bare=False):
        """'your' / 'my' / "Sam's" / 'a' - how to say whose a thing is."""
        if t.owner == self.YOU:
            return "yours" if bare else "your"
        if t.owner == self.ME:
            return "mine" if bare else "my"
        if t.owner and t.owner in self.things:
            return self.things[t.owner].name + "'s"
        return "nobody's in particular" if bare else "a"

    def _traits(self, t):
        bits = [t.props[a] for a in ("size", "colour") if a in t.props] + t.props.get("traits", [])
        return bits

    def describe(self, key_or_thing):
        """One or two sentences about a thing from its object file, or None if the file is empty."""
        t = key_or_thing if isinstance(key_or_thing, Thing) else self.things.get(str(key_or_thing).lower())
        if t is None:
            return None
        if t.key == self.YOU:
            return self.describe_you()
        if t.key == self.ME:
            return None
        parts = []
        subj = t.name[:1].upper() + t.name[1:]
        if t.kind:
            det = self.whose(t)
            det = det if det != "a" else ("an" if t.kind[0] in "aeiou" else "a")
            parts.append(f"{subj} is {det} {t.kind}" if not t.name.lower().startswith(("your ", "my ")) else
                         f"{subj} is {'yours' if t.owner == self.YOU else 'mine'}")
        elif t.owner:
            parts.append(f"{subj} is {self.whose(t, bare=True)}")
        traits = self._traits(t)
        if traits:
            parts.append(("it's " if parts else f"{subj} is ") + " and ".join(traits))
        if t.has:
            had = [self.things[h] for h in t.has if h in self.things]
            parts.append(("it has " if parts else f"{subj} has ") + ", ".join(self._a(h) for h in had))
        if t.where and t.where in self.things:
            place = self.things[t.where].name
            animate = t.is_kind("person") or any(t.is_kind(k) for k in ANIMATE)
            verb = "lives" if animate and t.at in ("in", "near") else "is"
            parts.append(("it " if parts else f"{subj} ") + f"{verb} {t.at} {place}")
        if "from" in t.props:
            parts.append(("it's " if parts else f"{subj} is ") + f"from {t.props['from']}")
        if not parts:
            return None
        text = parts[0] + (" - " + ", ".join(parts[1:]) if len(parts) > 1 else "") + "."
        if t.source == "read":
            text += " (I read that.)"
        return text

    def _a(self, t):
        # "a brown dog", not "an asleep dog": only what a thing looks like goes in front of it
        bits = [t.props[a] for a in ("size", "colour") if a in t.props] + [t.kind or t.name]
        return ("an " if bits[0][0] in "aeiou" else "a ") + " ".join(bits)

    def describe_you(self):
        you = self.things[self.YOU]
        parts = []
        name = you.props.get("name")
        kinds = [k for k in you.kinds if k != "person"]
        what = " and ".join(("an " if k[0] in "aeiou" else "a ") + k +
                            next((f" {pk.split()[-1]} {v}" for pk, v in you.props.items()
                                  if pk.startswith(k + " ")), "") for k in kinds)
        if name:
            parts.append(f"You're {name}" + (f", {what}" if what else ""))
        else:
            parts.append((f"You're {what}" if what else "You're the person I'm talking to") +
                         " - you haven't told me your name")
        if you.where and you.where in self.things:
            place = self.things[you.where].name
            parts.append(f"you live {you.at} {place}" if you.at in ("in", "near") else f"you're {you.at} {place}")
        if "from" in you.props:
            parts.append(f"you're from {you.props['from']}")
        yours = [t for t in self.things.values() if t.owner == self.YOU]
        if yours:
            parts.append("you have " + self._list(yours))
        if self.feelings:
            turn, how = self.feelings[-1]
            parts.append(f"earlier you said you were {how}")
        return parts[0] + (", " + ", ".join(parts[1:]) if len(parts) > 1 else "") + "."

    def _list(self, things):
        """'a dog, Rex, and a cat, Luna' / 'three dogs: Rex, Max and Milo'."""
        by_kind = {}
        for t in things:
            by_kind.setdefault(t.kind or "thing", []).append(t)
        out = []
        for kind, ts in by_kind.items():
            named = [t.name for t in ts if not t.name.lower().startswith(("your ", "my "))]
            if len(ts) == 1:
                t = ts[0]
                out.append(f"{self._a(t)}, {t.name}" if named else self._a(t))
            else:
                words = {2: "two", 3: "three", 4: "four", 5: "five"}.get(len(ts), str(len(ts)))
                out.append(f"{words} {kind}s" + (f": {self._join(named)}" if named else ""))
        return self._join(out)

    @staticmethod
    def _join(xs):
        xs = list(xs)
        return xs[0] if len(xs) == 1 else ", ".join(xs[:-1]) + " and " + xs[-1] if xs else ""

    def summary(self, agent):
        """What its world holds: you, itself, and everything else - grouped the way a person would list it."""
        others = [t for t in self.things.values() if t.key not in (self.YOU, self.ME, "home")]
        known = [t for t in others if t.kind or t.owner or t.props or t.has or t.where]
        relations = sum(len(m.episodes) for m in agent.maps.maps.values())
        if not others:
            return ("My world is just the two of us so far - you, and me on " + self.things["home"].name +
                    ". I start empty: tell me about something, or ask me about a place and I'll go and read.")
        lines = [f"My world so far: you, me, and {len(others)} other thing{'s' if len(others) != 1 else ''}."]
        yours = [t for t in known if t.owner == self.YOU]
        if yours:
            lines.append("Yours: " + self._list(yours) + ".")
        places = [t for t in others if t.is_kind("place") or t.is_kind("city") or t.is_kind("country")
                  or any(t.where == x.key for x in []) or t.key in {o.where for o in others if o.where}]
        places = list(dict.fromkeys(places))
        if places:
            lines.append("Places: " + self._join(p.name for p in places).rstrip(".") + ".")
        read = [t for t in others if t.source == "read" and t not in places]
        if read:
            lines.append("Things I read about: " + self._join(t.name for t in read[:8]).rstrip(".") + ".")
        rest = [t for t in others if t not in yours and t not in places and t not in read]
        if rest:
            kinds = {}
            for t in rest:
                kinds.setdefault(t.kind or "", []).append(t.name)
            bits = [f"{self._join(v)}" + (f" ({k}{'s' if len(v) > 1 else ''})" if k else "") for k, v in kinds.items()]
            lines.append("Also: " + "; ".join(bits).rstrip(".") + ".")
        if relations:
            spaces = [agent.maps._dim(s) if hasattr(agent.maps, "_dim") else s for s in agent.maps.maps]
            lines.append(f"I know {relations} relation{'s' if relations != 1 else ''} between them "
                         f"({', '.join(spaces)}), and I can work out ones you never told me.")
        return " ".join(lines)

    # ------------------------------------------------------------------ interoception
    def feel(self, agent):
        """How it is, read off its own state - the brain's interoception, for a brain whose body is its stores.

        Sleep pressure is facts written but not yet consolidated (adenosine, more or less); following is how many
        of the last few turns it could ground; being corrected lately is what stings. None of it is a mood it was
        given - each clause is a reading, and it changes when the state does."""
        trace = agent.trace[-8:]
        pressure = agent.unslept / max(1, agent.sleep_every)
        grounded = sum(1 for t in trace if t.get("grounded")) / len(trace) if trace else 1.0
        corrected = sum(1 for t in trace if t.get("intent") == "correction")
        facts = len(agent.maps.log)
        things = len([t for t in self.things.values() if t.key not in (self.YOU, self.ME, "home")])
        if not trace:
            return ("Fresh - I've only just woken up, and I hold nothing yet. Tell me about something and I'll "
                    "start building a world out of it.")
        bits = []
        if pressure < 0.34:
            bits.append("Good - clear-headed")
        elif pressure < 0.8:
            bits.append(f"Fine, though I'm carrying {agent.unslept} thing{'s' if agent.unslept != 1 else ''} I "
                        f"haven't slept on yet")
        else:
            bits.append(f"Getting full - {agent.unslept} things I haven't consolidated; I'll need to sleep soon")
        if grounded >= 0.8:
            bits.append("and I'm following you")
        elif grounded >= 0.5:
            bits.append("and mostly following you")
        else:
            lost = sum(1 for t in trace if not t.get("grounded"))
            bits.append(f"but a bit lost - I couldn't make sense of {lost} of the last {len(trace)} things you said")
        text = ", ".join(bits) + "."
        if corrected:
            text += f" You've corrected me {corrected} time{'s' if corrected != 1 else ''} just now, which I'm taking seriously."
        if things or facts:
            text += f" My world has {things} thing{'s' if things != 1 else ''} in it now."
        if self.feelings and self.feelings[-1][0] >= self.turn - 6:
            text += f" How about you - still {self.feelings[-1][1]}?"
        else:
            text += " How are you?"
        return text

    # ------------------------------------------------------------------ asking
    def question(self, sentence):
        """A question the object files can answer -> (kind, thing-or-None, extra), or None."""
        s = re.sub(r"\s+", " ", sentence.strip()).rstrip("?.! ").lower()
        s = re.sub(r"^(so|ok|okay|and|but|well|hey|nano)[, ]+", "", s)
        if re.match(r"^(how are you( (feeling|doing))?( today| now)?|how do you feel|how('s| is) it going|"
                    r"are you (ok|okay|alright|all right)|how are things|how you doing)$", s):
            return ("feel", None, None)
        if re.match(r"^(what('s| is) in your world|describe (your|the) world|what do you know about the world|"
                    r"what (things|objects) do you know( about)?|what exists|tell me about (your|the) world|"
                    r"what is your world( like)?|what does your world (look like|contain)|what do you know)$", s):
            return ("world", None, None)
        if re.match(r"^(where are you|where do you live|where are you from|where do you run|where are you running)$", s):
            return ("home", None, None)
        if re.match(r"^(who am i|what do you know about me|tell me about me|tell me about myself|"
                    r"what do you know about myself)$", s):
            return ("you", self.things[self.YOU], None)
        if re.match(r"^(what('s| is) my name|do you know my name|do you remember my name)$", s):
            return ("name", self.things[self.YOU], None)
        m = re.match(r"^(?:what do i (?:have|own)|what are my ([a-z]+?)s?|what ([a-z]+?)s? do i have|"
                     r"do i have (?:a|an|any) ([a-z]+?)s?)$", s)
        if m:
            return ("belongings", self.things[self.YOU], next((g for g in m.groups() if g), None))
        if re.match(r"^(where do i (live|stay|work)|where am i from|where('s| is) my home)$", s):
            return ("where", self.things[self.YOU], None)
        pats = [(r"^(?:whose (?:is )?(.+)|who owns (.+)|who does (.+) belong to|who is (.+) owned by)$", "owner"),
                (r"^(?:where does (.+) (?:live|stay|sleep)|where is (.+) from)$", "where"),
                (r"^(?:what colou?r is (.+)|what does (.+) look like|what is (.+) like|how big is (.+))$", "props"),
                (r"^(?:what does (.+) (?:have|own))$", "has"),
                (r"^(?:where is (.+))$", "where-soft")]
        for pat, kind in pats:
            m = re.match(pat, s)
            if m:
                t = self.find(next(g for g in m.groups() if g))
                if t is not None and (kind != "where-soft" or t.where):
                    return ("where" if kind == "where-soft" else kind, t, None)
        return None

    def answer(self, q, agent):
        """-> reply text, or None if the object files turn out not to hold it."""
        kind, t, extra = q
        if kind == "feel":
            return self.feel(agent)
        if kind == "world":
            return self.summary(agent)
        if kind == "home":
            return (f"I run on {self.things['home'].name}. I don't have a place in the world the way you do - my "
                    f"world is what you've told me and what I've read, and it lasts as long as this conversation.")
        if kind == "you":
            return self.describe_you()
        if kind == "name":
            name = t.props.get("name")
            return f"You're {name}." if name else "You haven't told me your name yet - what is it?"
        if kind == "belongings":
            yours = [x for x in self.things.values() if x.owner == self.YOU and (not extra or x.is_kind(extra))]
            if not yours:
                return (f"You haven't told me about any {extra}s of yours." if extra else
                        "You haven't told me about anything of yours yet.")
            return "You have " + self._list(yours) + "."
        if kind == "owner":
            if not t.owner:
                return f"You haven't told me whose {t.name} is."
            return f"{t.name[:1].upper() + t.name[1:]} is {self.whose(t, bare=True)}."
        if kind == "where":
            who = "You" if t.key == self.YOU else t.name[:1].upper() + t.name[1:]
            if t.where and t.where in self.things:
                verb = "live" if t.key == self.YOU else "lives"
                return f"{who} {verb} in {self.things[t.where].name}."
            if "from" in t.props:
                return f"{who} {'are' if t.key == self.YOU else 'is'} from {t.props['from']}."
            return None
        if kind == "props":
            traits = self._traits(t)
            return (f"{t.name[:1].upper() + t.name[1:]} is {' and '.join(traits)}." if traits else None)
        if kind == "has":
            had = [self.things[h] for h in t.has if h in self.things]
            return (f"{t.name[:1].upper() + t.name[1:]} has {self._join(self._a(h) for h in had)}." if had else None)
        return None

    # ------------------------------------------------------------------ graph
    def edges(self):
        """Owner, possession and whereabouts as graph edges for the knowledge view."""
        out = []
        for t in self.things.values():
            if t.owner and t.owner in self.things:
                out.append((t.key, t.owner, "belongs to"))
            if t.where and t.where in self.things and t.key != self.ME:
                out.append((t.key, t.where, "in"))
        return out

In [ ]:
%%writefile /kaggle/working/brain/chat/language.py
"""Controlled English for NeuroMoE's cognitive map. A hand-written parser - no language model - that turns a fixed set
of sentence patterns into facts ("teach") and questions ("ask"). Anything else is refused rather than guessed at.

Facts      Alice is taller than Bob.          The cafe is north of the park.      Dan is 2 blocks east of Eve.
           Bob is shorter than Alice.         The bank is north-east of the park. Cara is more careful than Dan.
Questions  Is Alice taller than Cara?         Who is taller, Alice or Dan?        Who is the tallest?
           Rank everyone by height.           Who is between Alice and Dan?       How much taller is Alice than Dan?
           What is 2 blocks north of the park?  Is the bank north of the cafe?    Where is the cafe relative to the bank?
           Tell me about Alice.
Commands   sleep | what do you know | forget everything | help
"""
import re

NUMBERS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
           "ten": 10, "a": 1, "an": 1}
NUM = r"(?:\d+|one|two|three|four|five|six|seven|eight|nine|ten|a|an)"

# comparative -> (axis, sign). The axis is a dimension of its own; sign says which way along it the word points.
COMPARATIVES = {
    "taller": ("height", 1), "shorter": ("height", -1), "older": ("age", 1), "younger": ("age", -1),
    "bigger": ("size", 1), "larger": ("size", 1), "smaller": ("size", -1), "heavier": ("weight", 1),
    "lighter": ("weight", -1), "faster": ("speed", 1), "slower": ("speed", -1), "richer": ("wealth", 1),
    "poorer": ("wealth", -1), "stronger": ("strength", 1), "weaker": ("strength", -1), "smarter": ("smarts", 1),
    "better": ("quality", 1), "worse": ("quality", -1), "hotter": ("heat", 1), "colder": ("heat", -1),
    "warmer": ("heat", 1), "cooler": ("heat", -1), "louder": ("loudness", 1), "quieter": ("loudness", -1),
    "happier": ("happiness", 1), "sadder": ("happiness", -1), "cheaper": ("price", -1), "longer": ("length", 1),
}
SUPERLATIVE_EXCEPTIONS = {"best": "better", "worst": "worse"}
NOUNS = {"height": "height", "age": "age", "size": "size", "weight": "weight", "speed": "speed", "wealth": "wealth",
         "strength": "strength", "price": "price", "length": "length", "heat": "heat", "temperature": "heat",
         "quality": "quality", "loudness": "loudness", "happiness": "happiness"}

# spatial words -> (axis, sign). Every spatial axis lives in one shared space, so routes may mix them.
# Left/right are read as on a map (right = east); in front/behind and above/below get axes of their own.
DIRECTIONS = {"north": ("north", 1), "south": ("north", -1), "east": ("east", 1), "west": ("east", -1),
              "above": ("up", 1), "below": ("up", -1), "under": ("up", -1), "over": ("up", 1),
              "left": ("east", -1), "right": ("east", 1), "behind": ("front", -1), "in front": ("front", 1)}
SPATIAL_AXES = ("north", "east", "up", "front")
SAY_AXIS = {"north": ("north", "south"), "east": ("east", "west"), "up": ("above", "below"),
            "front": ("in front", "behind")}


def space_of(axis):
    return "place" if axis in SPATIAL_AXES else axis


def positive_word(axis):
    """The word for one step up an axis: 'height' -> 'taller', 'careful' -> 'more careful'."""
    for word, (a, s) in COMPARATIVES.items():
        if a == axis and s == 1:
            return word
    return axis if axis.endswith("er") else "more " + axis


CONTRACTIONS = {"'s": " is", "'re": " are", "'m": " am", "'ve": " have", "'ll": " will", "n't": " not",
                "'d": " would"}
LEADING = re.compile(r"^(?:so|well|ok(?:ay)?|right|and|but|then|anyway|hmm+|oh|yeah|actually|i think|"
                     r"i reckon|i guess|i'd say)[,\s]+", re.I)


def expand(text):
    """"he's the smartest" is a sentence. Refusing it because of an apostrophe is not a principled position.

    Contractions and the little words people start sentences with carry no content for this system, and the
    grammar it does understand is hiding behind them - so both come off before anything is parsed.
    """
    out = text
    for short, long_ in CONTRACTIONS.items():
        out = re.sub(re.escape(short) + r"\b", long_, out, flags=re.I)
    previous = None
    while previous != out:                     # "so, well, who is the laziest" - strip them all
        previous = out
        out = LEADING.sub("", out).lstrip()
    return out


def clean(text):
    text = expand(text.strip()).strip().rstrip("?.!;").strip()
    return re.sub(r"\s+", " ", text)


def sentences(text):
    """Split a message into sentences, so a whole story can be taught in one go."""
    return [s.strip() for s in re.split(r"(?<=[.!?;])\s+|\n+", text) if clean(s)]


POSSESSIVE = r"(?:the|a|an|my|your|his|her|our|their|its)"


def name(text):
    """'the Cafe' -> ('cafe', 'the Cafe'): a lookup key and how to say it.

    "my dog Rex" is Rex: when a phrase introduces something by category and then names it, the name is what the
    speaker will use next time, so that is what it gets filed under.
    """
    shown = text.strip().strip(",").strip()
    key = re.sub(r"^" + POSSESSIVE + r"\s+", "", shown.lower()).strip()
    words = shown.split()
    # "my dog Rex" -> Rex, but "the Cafe" stays as it is: it takes a category word in between for the last word
    # to be a name being introduced rather than part of the thing's own name
    if len(words) > 2 and words[-1][:1].isupper() and not words[-2][:1].isupper():
        key, shown = words[-1].lower(), words[-1]
    return key, shown


def comparative(word):
    """Known comparatives, plus any unknown '-er' word as a dimension of its own (so 'nicer' just works)."""
    w = word.lower()
    if w in COMPARATIVES:
        return COMPARATIVES[w]
    if len(w) > 4 and w.endswith("er") and w not in ("other", "never", "under", "over", "either", "rather"):
        return (w, 1)
    return None


def superlative(word):
    w = word.lower()
    if w in SUPERLATIVE_EXCEPTIONS:
        return comparative(SUPERLATIVE_EXCEPTIONS[w])
    if len(w) > 4 and w.endswith("est"):
        stem = w[:-3]
        for guess in (stem + "er", stem[:-1] + "ier" if stem.endswith("i") else None, stem + "r"):
            if guess and guess in COMPARATIVES:
                return COMPARATIVES[guess]
        return comparative(stem + "er")
    return None


def count(text):
    if not text:
        return 1
    text = text.strip().lower()
    return int(text) if text.isdigit() else NUMBERS.get(text)


COMPASS = r"(north|south|east|west)(?:[- ]?(north|south|east|west))?"
_ONE = r"(?:" + NUM + r"\s+)?(?:(?:blocks?|steps?|miles?|km|meters?|units?|streets?)\s+)?(?:north|south|east|west)(?:[- ]?(?:north|south|east|west))?"
MOVES = r"(" + _ONE + r"(?:(?:\s+and\s+|,\s*)" + _ONE + r")*)"


def moves(text):
    """'2 blocks north-east' / 'north' / '3 north and 1 east' -> [(axis, count), ...] or None."""
    total = {}
    for part in re.split(r"\s+and\s+|,\s*", text.strip().lower()):
        m = re.fullmatch(r"(?:(" + NUM + r")\s+)?(?:(?:blocks?|steps?|miles?|km|meters?|units?|streets?)\s+)?"
                         + COMPASS, part.strip())
        if not m:
            return None
        n = count(m.group(1))
        for word in filter(None, (m.group(2), m.group(3))):
            axis, sign = DIRECTIONS[word]
            total[axis] = total.get(axis, 0) + sign * n
    return [(a, c) for a, c in total.items() if c] or None


def _cmp(word):
    """A comparative phrase ('taller', 'more careful', 'less careful') -> (axis, sign, word) or None."""
    m = re.fullmatch(r"(more|less) (\w+)", word, re.I)
    if m:
        return m.group(2).lower(), 1 if m.group(1).lower() == "more" else -1, word.lower()
    c = comparative(word)
    return (c[0], c[1], word.lower()) if c else None


SPATIAL_WORDS = "|".join(sorted(DIRECTIONS, key=len, reverse=True))
HEDGE = r"(?:a bit |much |slightly |a lot |way |far |even )?"


PARENS = re.compile(r"\([^)]*\)")
CLAUSE = re.compile(r"[;:]|,\s+(?:which|who|where|and is|although|though|while)\b")
CATEGORY = re.compile(r"^(.{2,60}?)\s+(?:is|was|are|were)\s+(?:an?|the)\s+"
                      r"((?:[a-z][a-z'-]*)(?:\s+[a-z][a-z'-]*){0,3})", re.I)
# a phrase that stops mid-way is not the name of anything: "genre of popular" and "synthesis of a" were filling
# the map with things no page will ever be about, and every one of them wasted a reading
DANGLING = ("of", "a", "an", "the", "and", "or", "in", "to", "for", "with", "from", "by", "as", "that", "which")
GENERIC = {"type", "kind", "form", "sort", "number", "member", "part", "group", "term", "word", "name",
           "example", "variety", "one", "thing", "way", "series", "set", "range", "list", "lists"}


def phrase(text):
    """Trim a captured noun phrase back to something that could be the name of a thing, or None."""
    words = re.sub(r"\s+", " ", text.strip().strip(",.;:")).split()
    while words and words[-1].lower() in DANGLING:
        words.pop()                     # "genre of popular" -> "genre of" -> "genre"
    while words and words[0].lower() in ("a", "an", "the"):
        words.pop(0)
    if not words or len(" ".join(words)) < 3:
        return None
    if words[-1].lower() in GENERIC and len(words) == 1:
        return None                     # "a type", "a form": true of everything, and about nothing
    if len(words) > 4:
        words = words[:4]              # a name is short; anything longer is a sentence being mistaken for one
        while words and words[-1].lower() in DANGLING:
            words.pop()                # trimming before the cut is not enough: the cut can leave a new dangle
    return " ".join(words) if words else None
PLACE_IN = re.compile(r"\b(?:is|was)\s+(?:an?|the)?\s*[a-z' -]*\b(?:in|from|of)\s+"
                      r"([A-Z][\w'-]*(?:\s+[A-Z][\w'-]*){0,2})")
DID = re.compile(r"^(.{2,50}?)\s+(?:has |had |have )?(released|recorded|starred in|founded|created|wrote|"
                 r"developed|originated in|began in|emerged (?:in|from)|performed|appeared in|won|led|"
                 r"pioneered|popularised|popularized|influenced|enabled|provided|borrows from|"
                 r"consists of|combines|includes|features|uses|evolved from|developed from|spread to|"
                 r"became|grew out of|drew on|is rooted in|is based on|is used in|is played on)"
                 r"\s+(?:an?|the|his|her|their)?\s*"
                 r"([A-Za-z][\w' -]{2,40})", re.I)
ROLE_AS = re.compile(r"^(.{2,60}?)\s+(?:is|was)\s+(?:best )?known (?:as|for)\s+(?:an?|the|his|her)?\s*"
                     r"((?:[a-z][a-z'-]*)(?:\s+[a-z][a-z'-]*){0,2})", re.I)


def prose_facts(sentence, limit=2):
    """Facts out of ordinary written English - what an encyclopedia sentence is actually claiming.

    The conversational patterns are deliberately strict: a person saying "Rex is older than Max" means exactly
    that, and guessing would be worse than refusing. Prose is different - it is not addressed to this system,
    it is long, and the parts it asserts are buried in apposition and parentheses. So the reading path strips
    the decoration and takes only the two claim shapes that survive it: what a thing *is*, and where it is.

    Everything this produces is marked as read rather than told, and is encoded more weakly, because a sentence
    that happened to fit a pattern is much weaker evidence than a person telling you something.
    """
    s = PARENS.sub(" ", sentence)
    s = CLAUSE.split(s)[0].strip()
    # a pronoun subject is the page's subject; keeping it lets the body of an article contribute, which is
    # where nearly everything beyond "X is a Y" actually lives
    s = re.sub(r"^(It|They|This|These|He|She)\b", lambda mm: mm.group(1).lower(), s)
    s = re.sub(r"\s+", " ", s)
    out = []
    m = CATEGORY.match(s)
    if m:
        subject = phrase(m.group(1))
        # "an American actor and rapper" is two claims, not one: a coordination is a list of categories, and
        # keeping them separate is what later lets one of them be the link to something else
        for part in re.split(r"\band\b|,", m.group(2)):
            category = phrase(part)
            if subject and category and len(subject) <= 60 and subject.lower() != category.lower():
                out.append({"kind": "fact", "a": name(subject), "b": name(category),
                            "moves": [("kind", 1)], "word": "a kind of", "how": "read"})
    m = ROLE_AS.match(s)
    if m and len(out) < limit:
        who, role = phrase(m.group(1)), phrase(m.group(2))
        if who and role:
            out.append({"kind": "fact", "a": name(who), "b": name(role),
                        "moves": [("kind", 1)], "word": "known as", "how": "read"})
    m = PLACE_IN.search(s)
    if m and out and len(out) < limit:
        where = phrase(m.group(1))
        if where and where.lower() != out[0]["a"][0]:
            out.append({"kind": "fact", "a": out[0]["a"], "b": name(where),
                        "moves": [("origin", 1)], "word": "from", "how": "read"})
    # what a thing does or made: the relations that actually connect people to fields, which categories alone
    # never will - "Will Smith released an album" is what puts him anywhere near music
    m = DID.search(s)
    if m and len(out) < limit and not re.search(r"\b(is|was|are|were)\b", m.group(1)):
        who, what = phrase(m.group(1)), phrase(re.split(r"\band\b", m.group(3))[0])
        if who and what and who.lower() != what.lower():
            out.append({"kind": "fact", "a": name(who), "b": name(what),
                        "moves": [("involved", 1)], "word": m.group(2).lower(), "how": "read"})
    return out[:limit]


def parse(text):
    """-> dict with 'kind' in {fact, ask, command, unknown} and whatever that kind needs."""
    t = clean(text)
    low = t.lower()

    if low in ("sleep", "consolidate", "go to sleep", "rest", "dream"):
        return {"kind": "command", "do": "sleep"}
    if low in ("forget everything", "reset", "clear", "start over") or re.fullmatch(
            r"(?:(?:ok(?:ay)?|so|alright|well|then|please)[,!.]?\s+)*(?:let'?s\s+|let is\s+|can we\s+|i want to\s+)?"
            r"(?:restart|reset|start (?:over|again|fresh)|begin again|clear everything|forget everything)[.!?]*", low):
        return {"kind": "command", "do": "reset"}
    if low in ("what do you know", "show the map", "show map", "status", "what do you remember", "summary"):
        return {"kind": "command", "do": "summary"}
    if low in ("help", "what can i say", "examples", "what can you do"):
        return {"kind": "command", "do": "help"}

    # questions -------------------------------------------------------------------------------------------
    m = re.fullmatch(r"is (.+?) " + HEDGE + r"((?:more|less) \w+|\w+) than (.+)", t, re.I)
    if m and _cmp(m.group(2)):
        axis, sign, word = _cmp(m.group(2))
        return {"kind": "ask", "q": "compare", "a": name(m.group(1)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"is (.+?) (?:directly |exactly )?" + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(2)):
        exact = bool(re.match(r"\s*" + NUM + r"\s", m.group(2), re.I) and not re.match(r"\s*an?\s", m.group(2), re.I))
        return {"kind": "ask", "q": "direction", "a": name(m.group(1)), "b": name(m.group(3)),
                "moves": moves(m.group(2)), "exact": exact, "word": m.group(2).lower()}
    m = re.fullmatch(r"is (.+?) (?:to the |directly |just )?(" + SPATIAL_WORDS + r")(?: of)? (.+)", t, re.I)
    if m:
        axis, sign = DIRECTIONS[m.group(2).lower()]
        return {"kind": "ask", "q": "direction", "a": name(m.group(1)), "b": name(m.group(3)),
                "moves": [(axis, sign)], "exact": False, "word": m.group(2).lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is ((?:more|less) \w+|\w+),? (.+?),? or (.+)", t, re.I)
    if m and _cmp(m.group(1)):
        axis, sign, word = _cmp(m.group(1))
        return {"kind": "ask", "q": "which", "a": name(m.group(2)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"is (.+?) the (most|least) (\w+)(?: one)?", t, re.I)
    if m:
        return {"kind": "ask", "q": "extreme", "a": name(m.group(1)), "axis": m.group(3).lower(),
                "sign": 1 if m.group(2).lower() == "most" else -1, "word": f"{m.group(2)} {m.group(3)}".lower()}
    m = re.fullmatch(r"is (.+?) the (\w+)(?: one)?", t, re.I)
    if m and superlative(m.group(2)):
        axis, sign = superlative(m.group(2))
        return {"kind": "ask", "q": "extreme", "a": name(m.group(1)), "axis": axis, "sign": sign,
                "word": m.group(2).lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is the (most|least) (\w+)(?: one)?", t, re.I)
    if m:
        return {"kind": "ask", "q": "extreme", "axis": m.group(2).lower(),
                "sign": 1 if m.group(1).lower() == "most" else -1, "word": f"{m.group(1)} {m.group(2)}".lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is the (\w+)(?: one)?", t, re.I)
    if m and superlative(m.group(1)):
        axis, sign = superlative(m.group(1))
        return {"kind": "ask", "q": "extreme", "axis": axis, "sign": sign, "word": m.group(1).lower()}
    m = re.fullmatch(r"(?:rank|order|sort|list)(?: everyone| everything| them| all| the \w+)?(?: by| from)? (.+)", t,
                     re.I)
    if m:
        what = re.sub(r"\s+to\s+\w+$", "", m.group(1).lower().strip())  # "tallest to shortest" -> "tallest"
        found = None
        if what in NOUNS:
            found = (NOUNS[what], 1)
        elif re.fullmatch(r"(most|least) (\w+)", what):
            mm = re.fullmatch(r"(most|least) (\w+)", what)
            found = (mm.group(2), 1 if mm.group(1) == "most" else -1)
        else:
            found = superlative(what) or comparative(what) or ((what, 1) if re.fullmatch(r"\w+", what) else None)
        if found:
            return {"kind": "ask", "q": "rank", "axis": found[0], "sign": found[1]}
    m = re.fullmatch(r"(?:who|what)(?: is| are|'s)? between (.+?) and (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "between", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"how (?:much|many \w+|far) ((?:more|less) \w+|\w+) is (.+?) than (.+)", t, re.I)
    if m and _cmp(m.group(1)):
        axis, sign, word = _cmp(m.group(1))
        return {"kind": "ask", "q": "howmuch", "a": name(m.group(2)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"(?:what|who|which \w+)(?: is|'s| lies)? " + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(1)):
        return {"kind": "ask", "q": "at", "a": name(m.group(2)), "moves": moves(m.group(1)), "word": m.group(1).lower()}
    m = re.fullmatch(r"where is (.+?)(?: relative to| compared to| from| in relation to) (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "where", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"how far is (.+?) from (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "where", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"where is (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "about", "a": name(m.group(1))}
    # "is Rex a mammal", "are dogs animals": a question about category membership, which is only answerable
    # by chaining - Rex is a dog, a dog is a mammal - and so belongs to the reasoning loop rather than to a
    # single retrieval. Answering it from one hop is how "is Rex a mammal" came back "Rex is a dog".
    m = re.fullmatch(r"(?:is|are|was|were)\s+(.+?)\s+(?:an?|the)\s+(?:kind|type|sort)\s+of\s+(.+)", t, re.I) or \
        re.fullmatch(r"(?:is|are|was|were)\s+(.+?)\s+(?:an?)\s+(.+)", t, re.I) or \
        re.fullmatch(r"(?:is|are|was|were)\s+(.+?)\s+(\w+s)\b", t, re.I)
    if m and len(m.group(1).split()) <= 4 and len(m.group(2).split()) <= 3:
        return {"kind": "ask", "q": "member", "a": name(m.group(1).strip(" ?.")),
                "b": name(m.group(2).strip(" ?."))}
    # asking how two things connect: the question this system answers as a route through the map
    m = re.fullmatch(r"(?:what(?:'s| is)? the )?(?:relation(?:ship)?|connection|link) between (.+?) and (.+)", t,
                     re.I)
    if m:
        return {"kind": "ask", "q": "relate", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"(?:how (?:does|did|do|is) )?(.+?) (?:relate[sd]? to|connected to|affect(?:ed|s)?|"
                     r"influence[sd]?|shape[sd]?|bears? on|bore on|ha[sd] to do with|"
                     r"compare[sd]? (?:to|with)|inform[sd]?) (.+)", t, re.I)
    if m and len(m.group(1).split()) <= 8:
        return {"kind": "ask", "q": "relate", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"(?:write |give me |produce )?(?:an? )?(?:\w+ )*?(?:analysis|essay|account|piece) of (.+?) "
                     r"on (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "relate", "a": name(m.group(1)), "b": name(m.group(2)), "asked_for": "essay"}
    # "google what a platypus is", "look up Kuala Lumpur", "search for jazz history": an explicit request to go
    # and read. It was falling through to "I don't know anything about please", which is refusing an
    # instruction by misreading it as a claim.
    m = re.fullmatch(r"(?:please\s+|can you\s+|could you\s+|would you\s+|go\s+(?:and\s+)?)*"
                     r"(?:google|search(?:\s+(?:for|up|the web for|online for))?|look\s+up|look\s+into|"
                     r"find\s+out(?:\s+about)?|find\s+(?:info|information)\s+(?:on|about)|research|check)\s+"
                     r"(?:what|who|where|when|how|why|whether|if)?\s*(?:an?\s+|the\s+)?(.+?)"
                     r"(?:\s+(?:is|are|was|were|means?|does|do))?\s*[?.!]*", t, re.I)
    if m and m.group(1).strip():
        return {"kind": "ask", "q": "about", "a": name(m.group(1).strip(" ?.")), "lookup": True, "question": t}
    # where something is, asked about something the person has not placed for us: "where is Kuala Lumpur",
    # "what country is Beijing in", "which state is Philadelphia in". This is a question for the reader, not
    # for the map, and the answer is a place - so it is marked as one, and the reply leads with the place
    # rather than with a definition that mentions it somewhere in the middle.
    m = re.fullmatch(r"(?:where(?:'s| is| are)|where abouts is) (.+?)(?: located| situated)?", t, re.I) or \
        re.fullmatch(r"(?:what|which|in what|in which) (country|state|city|continent|region|province) "
                     r"(?:is|are) (.+?) (?:in|located in|part of)", t, re.I)
    if m and not re.search(r"\b(of|from|north|south|east|west)\b", t.split(" is ")[-1] if " is " in t else "",
                           re.I):
        target = m.group(m.lastindex)
        if len(target.split()) <= 5:
            unit = m.group(1).lower() if m.lastindex == 2 else "place"
            return {"kind": "ask", "q": "about", "a": name(target), "asked": "where", "unit": unit}
    m = re.fullmatch(r"(?:what do you know about|tell me about|describe|who is|what is) (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "about", "a": name(m.group(1))}

    # an instruction is a thing to do, not a thing to look up. "Write an essay on X" was being matched as a
    # question about the phrase and answered by recalling that somebody had said it, which is the most annoying
    # failure a system can have: it heard the request and filed it instead of obeying it.
    m = re.fullmatch(r"(?:please\s+)?(?:can you|could you|would you|will you)?\s*"
                     r"(write|compose|draft|produce|give me|make me|put together|do)\s+"
                     r"(?:me\s+)?(?:an?|the|some)?\s*"
                     # "a short essay", "a quick write-up", "a proper analysis": an adjective between the
                     # article and the thing asked for was enough to make this miss, and a missed instruction
                     # is filed as a remark and read back three turns later, which is the failure that started
                     # all of this. Let a couple of modifiers through.
                     r"(?:[a-z-]+\s+){0,3}?(essay|piece|write-?up|summary|account|note|paragraph|"
                     r"explanation|analysis|report)\s*(?:on|about|of|for)?\s*(.*)", t, re.I)
    if m:
        target = re.sub(r"^(the|a|an)\s+", "", m.group(3).strip(" .?"), flags=re.I)
        # "an essay on just KL": the qualifier says how narrowly to write, and is not part of the name
        target = re.sub(r"^(just|only|simply|purely)\s+", "", target, flags=re.I)
        anything = not target or re.fullmatch(r"any(thing| topic| subject)?|whatever|something", target, re.I)
        return {"kind": "instruct", "deliverable": m.group(2).lower().replace("write-up", "writeup"),
                "topic": None if anything else name(target), "said": t}
    m = re.fullmatch(r"(?:please\s+)?(?:can you|could you)?\s*(explain|summarise|summarize|describe|outline|"
                     r"list)\s+(?:me\s+)?(?:an?|the)?\s*(?:what you know about\s+)?(.+)", t, re.I)
    if m:
        kind = {"list": "list", "summarise": "summary", "summarize": "summary",
                "outline": "summary"}.get(m.group(1).lower(), "explanation")
        return {"kind": "instruct", "deliverable": kind, "topic": name(m.group(2).strip(" .?")), "said": t}

    # "X is the tallest" is a fact about X and everyone else, not the introduction of a thing called tallest
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) (?:the )?(most|least) (\w+)(?: of .+)?", t, re.I)
    if m:
        return {"kind": "fact", "a": name(m.group(1)), "extreme": True, "axis": m.group(3).lower(),
                "sign": 1 if m.group(2).lower() == "most" else -1, "word": f"{m.group(2)} {m.group(3)}".lower()}
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) (?:the )?(\w+est)(?: of .+| one)?", t, re.I)
    if m and superlative(m.group(2)):
        axis, sign = superlative(m.group(2))
        return {"kind": "fact", "a": name(m.group(1)), "extreme": True, "axis": axis, "sign": sign,
                "word": m.group(2).lower()}
    # "Rex, Max and Milo are dogs" / "I have three dogs: Rex, Max and Milo" - several things introduced at once,
    # which is how anyone actually introduces a set of things
    m = re.fullmatch(r"(?:i (?:have got|have|got|own|keep)|there (?:are|is)) "
                     r"(?:\d+ |one |two |three |four |five |six |a |an |some |several |many )?"
                     r"([a-z][\w' -]*?)s?[:,]? (?:called |named )?(.+)", t, re.I)
    if m and re.search(r",| and ", m.group(2)):
        members = [name(x) for x in re.split(r",| and ", m.group(2)) if x.strip()]
        if len(members) > 1:
            return {"kind": "fact", "group": members, "b": name(m.group(1)), "moves": [("kind", 1)],
                    "word": "a kind of", "mine": True}
    m = re.fullmatch(r"(.+?) (?:are|were) (?:a |an )?([a-z][\w' -]*)", t, re.I)
    if m and re.search(r",| and ", m.group(1)):
        members = [name(x) for x in re.split(r",| and ", m.group(1)) if x.strip()]
        if len(members) > 1:
            return {"kind": "fact", "group": members, "b": name(re.sub(r"s$", "", m.group(2))),
                    "moves": [("kind", 1)], "word": "a kind of"}

    # facts -----------------------------------------------------------------------------------------------
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) " + HEDGE + r"((?:more|less) \w+|\w+) than (.+)", t, re.I)
    if m and _cmp(m.group(2)):
        axis, sign, word = _cmp(m.group(2))
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": [(axis, sign)], "word": word}
    m = re.fullmatch(r"(.+?) (?:is|are) (?:directly |just )?" + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(2)):
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": moves(m.group(2)),
                "word": m.group(2).lower()}
    m = re.fullmatch(r"(.+?) (?:is|are) (?:directly |just |to the )?(" + SPATIAL_WORDS + r")(?: of)? (.+)", t, re.I)
    if m:
        axis, sign = DIRECTIONS[m.group(2).lower()]
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": [(axis, sign)],
                "word": m.group(2).lower()}
    # "Rex is a dog" - category rather than comparison. Last, so it never shadows "Rex is older than Max".
    # This is also the one pattern that fires often in encyclopedia prose, which is how anything read from the
    # web gets into the map at all.
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) (?:an?|the) ([a-z][\w' -]*)", t, re.I)
    if m and len(m.group(1).split()) <= 5 and "than" not in m.group(2):
        return {"kind": "fact", "a": name(m.group(1)), "b": name(re.sub(r"^(a|an|the) ", "", m.group(2))),
                "moves": [("kind", 1)], "word": "a kind of"}
    # "dogs are mammals" - the same claim in the plural, where English drops the article. Without this the
    # second half of every chain went unheard: "Rex is a dog" landed and "dogs are mammals" did not.
    m = re.fullmatch(r"([a-z][\w' -]*?)\s+are\s+([a-z][\w' -]*)", t, re.I)
    if m and len(m.group(1).split()) <= 4 and len(m.group(2).split()) <= 3 \
            and not re.search(r"\b(than|there|here|you|they|we|it|these|those|what|who)\b", t, re.I):
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(2)),
                "moves": [("kind", 1)], "word": "a kind of"}

    return {"kind": "unknown"}

In [ ]:
%%writefile /kaggle/working/brain/tests/check_language.py
"""Checks for brain/learn_language.py - run inside the Kaggle notebook before anything is trained."""
import os
import sys

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain import learn_language as LL  # noqa: E402
from brain import onecortex as O  # noqa: E402

failed = []


def check(name, ok, detail=""):
    print(("ok   " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""), flush=True)
    if not ok:
        failed.append(name)


# the delta rule learns a separable problem from dense input, from sparse input, and from both
g = torch.Generator().manual_seed(0)
X = torch.randn(3000, 20, generator=g)
w = torch.randn(3, 20, generator=g)
y = (X @ w.T).argmax(1)
m = LL.Delta(20, 3, 0.1, 0.0, seed=0, sparse=False).fit(X, None, y)
acc = float((m.proba(X, None).argmax(1) == y).float().mean())
check("delta rule learns dense", acc > 0.9, f"acc {acc:.3f}")
S = [[int(c) * 7 + 1, 1000 + int(c)] for c in y.tolist()]
m = LL.Delta(0, 3, 0.1, 1e-5, seed=0).fit(None, S, y)
acc = float((m.proba(None, S).argmax(1) == y).float().mean())
check("delta rule learns sparse (with decay)", acc > 0.99, f"acc {acc:.3f}")
m = LL.Delta(20, 3, 0.1, 0.0, seed=0).fit(X, S, y)
acc = float((m.proba(X, S).argmax(1) == y).float().mean())
check("delta rule learns both", acc > 0.95, f"acc {acc:.3f}")
gain = LL.balanced_gain(torch.tensor([0] * 90 + [1] * 10), 2)
check("rare class gets more gain", gain[1] > gain[0] * 5, str(gain.tolist()))

# the codes are stable across processes (crc32, not Python's salted hash) and in range
f1, f2 = LL.sentence_features("What do you mean?"), LL.sentence_features("What do you mean?")
check("sentence code is deterministic", f1 == f2 and all(0 <= i < LL.DIM for i in f1))
check("token code in range", all(0 <= i < LL.DIM for i in LL.token_features(["i", "have", "a", "dog"], 3)))

# switchboard tags and text
check("swda tags", [LL.SWDA.get(t) for t in ("sd", "qy", "br", "ft", "fc", "ad")] ==
      ["statement", "question", "meaning", "thanks", "bye", "directive"])
base = lambda t: __import__("re").split(r"[\^(,;]", t)[0].strip()
check("swda modifiers stripped", LL.SWDA.get(base("qy^d")) == "question" and LL.SWDA.get(base("+")) is None)
check("swda markup cleaned", LL.clean_swda("{F Uh, } I think [ it, + it's ] fine. / <laughter>") ==
      "Uh, I think it, it's fine.", LL.clean_swda("{F Uh, } I think [ it, + it's ] fine. / <laughter>"))

# objects
check("object words normalised", LL.norm_words("two dogs") == ["two", "dog"] and LL.norm_words("the cats") == ["cat"])
check("object labels", LL.object_labels(O.words("i have two dogs ."), "dog") == [0, 0, 0, 1, 0])
check("best run", LL.best_run([0.1, 0.9, 0.8, 0.2]) == (1, 3) and LL.best_run([0.1, 0.3, 0.2]) == (1, 2))
check("detokenise", LL.detok("i don 't like cats .") == "i don't like cats.", LL.detok("i don 't like cats ."))

# the rules being replaced, mapped as intended
got = {t: LL.rule_act(t) for t in ("huh?", "thanks!", "who wrote Hamlet?", "Rex is older than Max.", "hello")}
check("rules: acts", got == {"huh?": "meaning", "thanks!": "thanks", "who wrote Hamlet?": "question",
                            "Rex is older than Max.": "statement", "hello": "greet"}, str(got))
rels = ["have_pet", "live_in_citystatecountry", "has_profession", "other"]
got = [LL.rule_fact(s, rels) for s in ("i have two dogs.", "i live in Texas.", "i am a teacher.", "i like pizza.")]
check("rules: facts", got == [("have_pet", "dog"), ("live_in_citystatecountry", "Texas"),
                              ("has_profession", "teacher"), (None, None)], str(got))

# statistics
h = LL.holm({"a": {"p": 0.001, "direction": False}, "b": {"p": 0.01, "direction": True},
             "c": {"p": 0.5, "direction": True}})
check("holm continues past a wrong-direction rejection", not h["a"]["supported"] and h["b"]["supported"]
      and not h["c"]["supported"])
b = LL.bootstrap_diff(["x", "y"] * 20, ["x", "y"] * 20, ["x", "x"] * 20, LL.macro_f1, n=200)
check("bootstrap sees a real difference", b["diff"] > 0 and b["p"] < 0.05, str(b))

# the cortex reads, and token states line up with the words
net = O.OneCortex(40, embed=8, hidden=16, mem=4, leak=(0.5, 0.9))
voc = O.Vocab.__new__(O.Vocab)
voc.itos = O.SPECIAL + ["i", "have", "a", "dog", "."]
voc.stoi = {w: i for i, w in enumerate(voc.itos)}
X, tok = LL.cortex_codes(net, voc, ["i have a dog .", "i have"], tokens=True)
check("cortex codes shape", tuple(X.shape) == (2, 48) and tok[0].shape[0] == 5 and tok[1].shape[0] == 2,
      f"{tuple(X.shape)} {[t.shape for t in tok]}")

print("\n" + ("ALL CHECKS PASSED" if not failed else f"{len(failed)} FAILED: {failed}"))
sys.exit(1 if failed else 0)

In [ ]:
%%writefile /kaggle/working/brain/tests/nano_acts.jsonl
{"text": "yo!", "act": "greet"}
{"text": "hi nano!", "act": "greet"}
{"text": "hello", "act": "greet"}
{"text": "hey there", "act": "greet"}
{"text": "good morning", "act": "greet"}
{"text": "hi?", "act": "greet"}
{"text": "how are you today", "act": "question"}
{"text": "how are you doing", "act": "question"}
{"text": "what is your favorite food?", "act": "question"}
{"text": "what is your name?", "act": "question"}
{"text": "ok but what name do you posess?", "act": "question"}
{"text": "didnt you just say your name was nano?", "act": "question"}
{"text": "why did you not respond \"nano\"?", "act": "question"}
{"text": "What is Nana Mouskouri?", "act": "question"}
{"text": "what is going on in dallas tx right now", "act": "question"}
{"text": "ok but what is happening in dallas right now", "act": "question"}
{"text": "who wrote Hamlet?", "act": "question"}
{"text": "Why is the sky blue?", "act": "question"}
{"text": "What is the capital city of Japan?", "act": "question"}
{"text": "how many people live there?", "act": "question"}
{"text": "Is Alice taller than Carol?", "act": "question"}
{"text": "Who is the tallest?", "act": "question"}
{"text": "Who is the oldest?", "act": "question"}
{"text": "what did we talk about?", "act": "question"}
{"text": "do you like books?", "act": "question"}
{"text": "what should I do?", "act": "question"}
{"text": "what's in your world?", "act": "question"}
{"text": "who am I?", "act": "question"}
{"text": "whose is Max?", "act": "question"}
{"text": "what do I have?", "act": "question"}
{"text": "any news about SpaceX today?", "act": "question"}
{"text": "is Tokyo bigger than Paris?", "act": "question"}
{"text": "where does Luna live?", "act": "question"}
{"text": "can you tell me who invented the telephone?", "act": "question"}
{"text": "what do you mean", "act": "meaning"}
{"text": "what do you mean?", "act": "meaning"}
{"text": "huh?", "act": "meaning"}
{"text": "what?", "act": "meaning"}
{"text": "I don't get it", "act": "meaning"}
{"text": "what are you talking about", "act": "meaning"}
{"text": "sorry, what?", "act": "meaning"}
{"text": "come again?", "act": "meaning"}
{"text": "that makes no sense", "act": "meaning"}
{"text": "I don't understand", "act": "meaning"}
{"text": "can you say that again?", "act": "meaning"}
{"text": "My dog Rex is older than Max.", "act": "statement"}
{"text": "Milo is older than Rex.", "act": "statement"}
{"text": "Alice is taller than Bob.", "act": "statement"}
{"text": "The cafe is north of the park.", "act": "statement"}
{"text": "I had a rough day at work.", "act": "statement"}
{"text": "my boss yelled at me in front of everyone.", "act": "statement"}
{"text": "I love pizza.", "act": "opinion"}
{"text": "My name is Eric.", "act": "statement"}
{"text": "I live in Los Angeles.", "act": "statement"}
{"text": "I'm a student at UCLA.", "act": "statement"}
{"text": "Rex is brown.", "act": "statement"}
{"text": "Rex has a red ball.", "act": "statement"}
{"text": "I have a cat named Luna.", "act": "statement"}
{"text": "I am tired.", "act": "statement"}
{"text": "I've been building you all day.", "act": "statement"}
{"text": "I think it's going to rain tomorrow.", "act": "opinion"}
{"text": "that's a terrible answer", "act": "opinion"}
{"text": "I think you're getting better", "act": "opinion"}
{"text": "pizza is the best food", "act": "opinion"}
{"text": "honestly that movie was boring", "act": "opinion"}
{"text": "that's cool.", "act": "backchannel"}
{"text": "ok", "act": "backchannel"}
{"text": "cool", "act": "backchannel"}
{"text": "i see", "act": "backchannel"}
{"text": "uh huh", "act": "backchannel"}
{"text": "wow", "act": "backchannel"}
{"text": "nice", "act": "backchannel"}
{"text": "yes", "act": "agree"}
{"text": "yeah that's right", "act": "agree"}
{"text": "exactly", "act": "agree"}
{"text": "I agree", "act": "agree"}
{"text": "sure, sounds good", "act": "agree"}
{"text": "no, that's wrong", "act": "disagree"}
{"text": "no", "act": "disagree"}
{"text": "actually no, Ollie is older than Rex", "act": "disagree"}
{"text": "that's not true", "act": "disagree"}
{"text": "I don't think so", "act": "disagree"}
{"text": "thanks!", "act": "thanks"}
{"text": "thank you so much", "act": "thanks"}
{"text": "thanks, that helps", "act": "thanks"}
{"text": "sorry", "act": "sorry"}
{"text": "sorry about that", "act": "sorry"}
{"text": "my bad", "act": "sorry"}
{"text": "I apologize", "act": "sorry"}
{"text": "bye", "act": "bye"}
{"text": "thanks, bye!", "act": "bye"}
{"text": "see you later", "act": "bye"}
{"text": "good night", "act": "bye"}
{"text": "ok lets restart", "act": "directive"}
{"text": "forget everything", "act": "directive"}
{"text": "go to sleep", "act": "directive"}
{"text": "tell me about Rex.", "act": "directive"}
{"text": "write an essay about Kuala Lumpur", "act": "directive"}
{"text": "look up the population of Tokyo", "act": "directive"}
{"text": "please explain why", "act": "directive"}
{"text": "rank everyone by height", "act": "directive"}

In [ ]:
%%writefile /kaggle/working/brain/PREREGISTRATION_learned_language.md
# Pre-registration: learning what nano's hand-written rules do

Written 2026-09-29, before any data below has been loaded or any model trained.

## Why

nano's *knowledge* is learned (facts placed on maps by local plasticity, the gate's choices, the talking
striatum, the one cortex), but how it *understands* a sentence is hand-written: dialogue-act patterns
(`brain/discourse.py`), the object grammar (`brain/world.py`), and dozens of routing regexes. Each failure the
user found was patched with another rule. This replaces two of those rule sets with learned readouts, and
tests whether the learned versions beat the rules they replace.

## The learning rule

Both tasks read a sentence with the **one cortex** (the e-prop checkpoint, conversation perplexity 94.5,
frozen) and a **sparse word code** (hashed words, word pairs and the sentence's first word and final mark:
a fixed, unlearned projection, like the input to a striatal column). A readout layer learns with the
**delta rule** (Widrow-Hoff / Rescorla-Wagner): each synapse changes by (target - output) x presynaptic
activity, with synaptic decay. That is a local rule: one layer, no error passed back through the cortex. It
is the striatum learning which cortical pattern means what.

The learning rate and decay are chosen on a validation split from the grid {0.01, 0.03, 0.1} x {0, 1e-5};
the tuned baseline gets the same treatment (C in {0.1, 0.3, 1, 3, 10}). Because macro-F1 counts rare acts
("meaning", "thanks") as much as statements, every learned arm and the tuned baseline weight each class's error
by its inverse training frequency (for the delta rule: a neuromodulatory gain on the error of rare outcomes;
for logistic regression: `class_weight="balanced"`). Dense cortex features are standardised with training-set
means and deviations. Readouts train for 4 passes over the training data in minibatches of 256, with the
learning rate falling linearly to zero.

## Arms

- `delta-words`: delta readout on the sparse word code only
- `delta-cortex`: delta readout on the cortex's states only
- `delta-both`: delta readout on both (the arm nano would use)
- `rules`: nano's current hand-written rules, mapped to the task's labels (the thing being replaced)
- `tfidf-lr`: tuned logistic regression on tf-idf word 1-2 grams (the standard yardstick, not brain-like)
- `majority`: the most frequent class

Five seeds for every learned arm (a seed sets shuffle order and minibatches).

## Task A: dialogue acts

**Data.** Switchboard Dialog Act corpus (github.com/cgpotts/swda), utterances with DAMSL tags, collapsed to
12 classes nano routes on: statement, opinion, question, backchannel, agree, disagree, thanks, sorry, greet,
bye, meaning (signal-non-understanding: "huh?", "what do you mean"), directive. Other tags and utterance
continuations ("+") are dropped. Split by conversation, seeded: 85% train / 5% validation / 10% test.

**Second test set.** `tests/nano_acts.jsonl`: utterances typed to nano in real sessions, plus others of the
same kinds, hand-labelled with the same 12 classes before any model saw them. This is the test that matters
for nano; Switchboard is phone conversation and differs from typed chat.

**Primary metric.** Macro-F1 (every class counts equally, so "meaning" and "thanks" matter as much as
statements).

- **H1.** `delta-both` beats `rules` on the nano set (difference in macro-F1 of the seed-averaged prediction,
  95% bootstrap interval over utterances excludes 0).
- **H2.** `delta-both` is within 0.03 macro-F1 of `tfidf-lr` on the Switchboard test set or better
  (non-inferiority: the mean over seeds minus the tuned baseline is > -0.03 by a one-sided t-test over seeds).
- **H3.** The cortex adds something: `delta-both` beats `delta-words` on Switchboard test macro-F1 (one-sided
  paired t-test over seeds).

## Task B: facts about the speaker

**Data.** Dialogue NLI (Welleck et al. 2019; huggingface.co/datasets/pietrolesci/dialogue_nli): persona and
dialogue sentences, each annotated with a triple (subject, relation, object), e.g. "i have two dogs." ->
(i, have_pet, dog). Unique sentences from train / dev / test; any test sentence also in train is removed.
Relations with fewer than 50 training sentences are merged into "other".

**What is learned.** The relation (a sentence-level readout) and the object (a token-level readout marking
which words are the object; the object is the highest-scoring contiguous run). Token features are the
cortex's state after the token and the sparse code of the token and its neighbours.

**Primary metric.** Triple exact match: relation correct and object words equal to the annotated object
(lower-cased, plural "s" and articles stripped). Also relation macro-F1 and object token F1.

- **H4.** `delta-both` beats `rules` (the object grammar of `brain/world.py`, mapped onto Dialogue NLI
  relations where it has one) on triple exact match on the test set (bootstrap as H1).
- **H5.** `delta-both` relation macro-F1 is within 0.03 of `tfidf-lr` or better (as H2).
- **H6.** The cortex helps find the object: `delta-both` beats `delta-words` on object token F1 (as H3).

## Correction and decision

Holm step-down over H1-H6 at alpha 0.05; the procedure continues past a rejection in the wrong direction.

nano switches a rule set to its learned readout only if that task's primary hypothesis (H1 for acts, H4 for
facts) is supported. Otherwise the rules stay, and the result is reported as it is.

## Known limits, stated in advance

- Dialogue NLI is about the speaker ("i ..."). A learned extractor from it covers facts about the person
  (pets, home, job, likes, family, age); facts about third parties ("Rex is brown") stay with the object
  grammar until there is labelled data for them.
- The cortex is frozen here. Its perplexity (94.5) is poor; if it improves, the readouts are retrained on it.
- The nano act test set is small (~100 utterances) and labelled by the person who wrote the rules. Its
  intervals will be wide.

## Result (run 1, eric847/brain-learned-language, commit 335352b, 2026-09-29)

Acts (Switchboard test macro-F1 / nano set macro-F1): delta-words 0.361 / 0.416, delta-cortex 0.383 / 0.506,
delta-both 0.384 / 0.508, rules 0.357 / 0.768, tfidf-lr 0.611 / 0.683, majority 0.050 / 0.020.
Facts (test triple exact / relation macro-F1 / object token F1): delta-words 0.027 / 0.127 / 0.404,
delta-cortex 0.043 / 0.120 / 0.439, delta-both 0.045 / 0.120 / 0.451, rules 0.006 (covers 5% of sentences),
tfidf-lr 0.202 / 0.350 / 0.616.

Holm: H3 supported (cortex adds to acts, +0.022), H4 supported (+0.040 over the object grammar), H6 supported
(cortex helps find the object, +0.047). H1, H2, H5 not supported.

Decision. Acts: H1 failed, the rules stay. Facts: H4 passed, but the pre-registered comparison was weak - the
object grammar covers only 5% of Dialogue NLI's sentences, and 4.5% exact triples is not usable (it files "i have
a dog named pedro" as have_sibling). The learned extractor is **not** deployed; this departs from the stated
decision rule, and that is why.

What went wrong in the design, found after the fact: the word readouts and every facts arm peaked at the top of
the learning-rate grid (0.1), so they were undertrained - the sparse updates are divided by batch size and by the
square root of the active count, which made them tiny. A delta rule on the same features converges to the same
solution as logistic regression, so most of the words-vs-LR gap is this, not the rule. The cortex arms peaked
inside the grid (0.03) for acts, so their ceiling (~0.385) is the cortex's features at perplexity 94.
Even the tuned logistic regression (0.683) loses to the rules (0.768) on the nano set: Switchboard phone speech
does not transfer well to typed chat.

## Amendment 1 (post hoc, 2026-09-29, before run 2)

Written after seeing run 1, so run 2 is a correction, not a confirmation: the learning-rate grid becomes
{0.03, 0.1, 0.3, 1, 3} x {0, 1e-5} and readouts train for 8 passes instead of 4. Everything else - data,
splits, arms, hypotheses, tests, Holm, the decision rule - is unchanged. The facts decision keeps the caveat
above: a learned extractor is deployed only if it is also usable in absolute terms, which run 1's was not.

### Checks

In [ ]:
r = subprocess.run([sys.executable, "tests/check_language.py"], capture_output=True, text=True, cwd=W)
print(r.stdout[-4000:], r.stderr[-3000:])
assert r.returncode == 0, "checks failed - not training"

### Pilot

In [ ]:
# a small real-data pilot first: the whole pipeline end to end on a slice, so a bug costs minutes, not hours
t0 = time.time()
r = subprocess.run([sys.executable, "-u", "brain/learn_language.py", "--task", "both", "--swda", DATA + "/swda.zip",
                    "--dnli", DATA + "/dnli", "--cortex", CORTEX, "--out", "/kaggle/working/pilot", "--limit", "3000"],
                   capture_output=True, text=True, cwd=W)
print(r.stdout[-6000:], r.stderr[-4000:])
assert r.returncode == 0, "pilot failed"
print(f"pilot took {time.time() - t0:.0f}s")

### The run

In [ ]:
t0 = time.time()
log = open("/kaggle/working/run.log", "w")
p = subprocess.Popen([sys.executable, "-u", "brain/learn_language.py", "--task", "both", "--swda", DATA + "/swda.zip",
                      "--dnli", DATA + "/dnli", "--cortex", CORTEX, "--out", "/kaggle/working/run"],
                     cwd=W, stdout=log, stderr=subprocess.STDOUT)
while p.poll() is None:
    time.sleep(120)
    lines = open("/kaggle/working/run.log").read().splitlines()
    print(f"[{(time.time() - t0) / 60:.0f} min]", *lines[-3:], sep="\n  ", flush=True)
print(open("/kaggle/working/run.log").read()[-8000:])
assert p.returncode == 0, "run failed"

### Verdicts

In [ ]:
R = json.load(open("/kaggle/working/run/results.json"))
for task in ("acts", "facts"):
    t = R[task]
    print(f"== {task}: sizes {t['sizes']}")
    for arm, v in t["arms"].items():
        print(f"  {arm}: " + ", ".join(f"{k} {v[k]}" for k in v if not isinstance(v[k], (list, dict))))
    for k in ("rules", "tfidf-lr", "majority"):
        if k in t:
            print(f"  {k}: " + ", ".join(f"{a} {b}" for a, b in t[k].items() if not isinstance(b, list)))
print(json.dumps(R["verdict"], indent=1))
print("\nnano set, where delta-both and the label differ:")
for text, gold, pred in R["acts"]["confusion_nano"]:
    print(f"  {gold:>11} <- {pred:<11} {text}")
print("\nfacts, first test sentences:")
for s, r, o, gr, go in R["facts"]["examples"][:25]:
    print(f"  {s!r}: said ({r}, {o!r}), annotated ({gr}, {go!r})")